
# LCES ASAP-8 — Paper-like transductive run với official rubrics

Notebook này chạy **LCES pairwise + RankNet** cho cả 8 essay sets của ASAP trong một pipeline tuần tự:

1. Load dữ liệu ASAP (`asap_train.csv`, `asap_val.csv`, `asap_test.csv`) hoặc Kaggle `training_set_rel3.tsv`.
2. Sample 10% essays cho từng essay set.
3. Sample `M=5000` ordered pairwise comparisons mỗi set.
4. Query LLM hai chiều `(essay_i, essay_j)` và `(essay_j, essay_i)` để debias position bias.
5. Train RankNet từ pairwise labels.
6. Map latent scores về score range chính thức và tính QWK/Spearman.

Default cố gắng bám setup paper:

- Pairwise LLM: `mistralai/Mistral-7B-Instruct-v0.2`
- Temperature: `0.1`
- Pairwise comparisons: `M=5000`
- RankNet: 100 epochs, batch size 4096, hidden 256, dropout 0.3, weight decay 0.01
- Rubric: official ReadMeFirst rubrics đã được nhúng sẵn vào notebook này
- Evaluation: random 10% essays mỗi prompt

Lưu ý: chạy đủ 8 sets với local LLM sẽ rất lâu. Notebook có cache/resume từng set nên có thể dừng rồi chạy tiếp.


In [1]:

# ============================================================
# 0. Install dependencies
# ============================================================
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers openai
!pip -q install -U huggingface_hub hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"


In [2]:

# ============================================================
# 1. Imports and global config
# ============================================================
import os
import re
import gc
import json
import math
import time
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


DEVICE: cuda


In [3]:

# ============================================================
# 2. Paths and experiment switches
# ============================================================
# Option A: prepared split files, same style as your original notebook.
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

# Option B: original Kaggle ASAP file. Used only if the three split files above do not exist.
ASAP_TSV_PATH = "/content/training_set_rel3.tsv"

OUTPUT_DIR = Path("/content/lces_asap8_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "judgments").mkdir(exist_ok=True)
(OUTPUT_DIR / "pairs").mkdir(exist_ok=True)
(OUTPUT_DIR / "subsets").mkdir(exist_ok=True)
(OUTPUT_DIR / "embeddings").mkdir(exist_ok=True)
(OUTPUT_DIR / "predictions").mkdir(exist_ok=True)

RUN_ESSAY_SETS = list(range(5, 9))

# Paper-like default: random 10% essays per prompt, M=5000 pairwise comparisons.
SAMPLE_FRAC = 0.10
M_PAIRWISE = 5000

# LLM generation setup.
LLM_BACKEND = "local_hf"  # this notebook implements local HF generation
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
GEN_TEMPERATURE = 0
MAX_NEW_TOKENS = 160
LLM_MAX_INPUT_TOKENS = 12000
PAIRWISE_BATCH_SIZE = 96
LOAD_IN_4BIT = False  # set True if you hit GPU OOM

# RankNet setup from paper appendix.
RANKNET_EPOCHS = 100
RANKNET_BATCH_SIZE = 4096
RANKNET_LR = 1e-3
RANKNET_WEIGHT_DECAY = 0.01
RANKNET_HIDDEN_DIM = 256
RANKNET_DROPOUT = 0.3

# Embedding setup.
# Paper main: text-embedding-3-large.
# If OPENAI_API_KEY is unavailable, the notebook falls back to RoBERTa-base [CLS].
EMBEDDING_BACKEND = "openai" if os.environ.get("OPENAI_API_KEY") else "roberta_cls"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
LOCAL_EMBEDDING_MODEL = "roberta-base"
LOCAL_EMBEDDING_MAX_LENGTH = 512
LOCAL_EMBEDDING_BATCH_SIZE = 16

# For debugging only. Leave False for full run.
QUICK_TEST = False
if QUICK_TEST:
    RUN_ESSAY_SETS = [1]
    SAMPLE_FRAC = 0.02
    M_PAIRWISE = 50
    RANKNET_EPOCHS = 3
    PAIRWISE_BATCH_SIZE = 2
    MAX_NEW_TOKENS = 64

EXPERIMENT_TAG = (
    f"{MODEL_ID.split('/')[-1].replace('.', '').replace('-', '_')}"
    f"_temp{str(GEN_TEMPERATURE).replace('.', '')}"
    f"_m{M_PAIRWISE}_seed{SEED}_frac{str(SAMPLE_FRAC).replace('.', '')}"
)

print("RUN_ESSAY_SETS:", RUN_ESSAY_SETS)
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("EMBEDDING_BACKEND:", EMBEDDING_BACKEND)
print("OUTPUT_DIR:", OUTPUT_DIR)


RUN_ESSAY_SETS: [5, 6, 7, 8]
EXPERIMENT_TAG: Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01
EMBEDDING_BACKEND: roberta_cls
OUTPUT_DIR: /content/lces_asap8_outputs


In [4]:
# ============================================================
# 3. Official ASAP prompts and rubrics extracted from ReadMeFirst docs
# ============================================================
import json

ASAP_TASKS = json.loads(r'''{
  "1": {
    "prompt": "Prompt\nMore and more people use computers, but not everyone agrees that this benefits society. Those who support advances in technology believe that computers have a positive effect on people. They teach hand-eye coordination, give people the ability to learn about faraway places and people, and even allow people to talk online with other people. Others have different ideas. Some experts are concerned that people are spending too much time on their computers and less time exercising, enjoying nature, and interacting with family and friends.\nWrite a letter to your local newspaper in which you state your opinion on the effects computers have on people. Persuade the readers to agree with you.",
    "rubric": "Rubric Guidelines\nScore Point 1: An undeveloped response that may take a position but offers no more than very minimal support. Typical elements:\nContains few or vague details.\nIs awkward and fragmented.\nMay be difficult to read and understand.\nMay show no awareness of audience.\nScore Point 2: An under-developed response that may or may not take a position. Typical elements:\nContains only general reasons with unelaborated and/or list-like details.\nShows little or no evidence of organization.\nMay be awkward and confused or simplistic.\nMay show little awareness of audience.\nScore Point 3: A minimally-developed response that may take a position, but with inadequate support and details. Typical elements:\nHas reasons with minimal elaboration and more general than specific details.\nShows some organization.\nMay be awkward in parts with few transitions.\nShows some awareness of audience.\nScore Point 4: A somewhat-developed response that takes a position and provides adequate support. Typical elements:\nHas adequately elaborated reasons with a mix of general and specific details.\nShows satisfactory organization.\nMay be somewhat fluent with some transitional language.\nShows adequate awareness of audience.\nScore Point 5: A developed response that takes a clear position and provides reasonably persuasive support. Typical elements:\nHas moderately well elaborated reasons with mostly specific details.\nExhibits generally strong organization.\nMay be moderately fluent with transitional language throughout.\nMay show a consistent awareness of audience.\nScore Point 6: A well-developed response that takes a clear and thoughtful position and provides persuasive support. Typical elements:\nHas fully elaborated reasons with specific details.\nExhibits strong organization.\nIs fluent and uses sophisticated transitional language.\nMay show a heightened awareness of audience."
  },
  "2": {
    "prompt": "Prompt\nCensorship in the Libraries\n\"All of us can think of a book that we hope none of our children or any other children have taken off the shelf. But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly the same right and so does everyone else. And then we have no books left on the shelf for any of us.\" --Katherine Paterson, Author\nWrite a persuasive essay to a newspaper reflecting your vies on censorship in libraries. Do you believe that certain materials, such as books, music, movies, magazines, etc., should be removed from the shelves if they are found offensive? Support your position with convincing arguments from your own experience, observations, and/or reading.",
    "rubric": "Rubric Guidelines—Domain 1: Writing Applications\nScore Point 6: A Score Point 6 paper is rare. It fully accomplishes the task in a thorough and insightful manner and has a distinctive quality that sets it apart as an outstanding performance.\nIdeas and Content\nDoes the writing sample fully accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it\npresent a unifying theme or main idea without going off on tangents?\nstay completely focused on topic and task?\nDoes the writing sample include thorough, relevant, and complete ideas? Does it\ninclude in-depth information and exceptional supporting details that are fully developed?\nfully explore many facets of the topic?\nOrganization\nAre the ideas in the writing sample organized logically? Does the writing\npresent a meaningful, cohesive whole with a beginning, a middle, and an end (i.e., include an inviting introduction and a strong conclusion)?\nprogress in an order that enhances meaning?\ninclude smooth transitions between ideas, sentences, and paragraphs to enhance meaning of text (i.e., have a clear connection of ideas and use topic sentences)?\nStyle\nDoes the writing sample exhibit exceptional word usage? Does it\ninclude vocabulary to make explanations detailed and precise, descriptions rich, and actions clear and vivid (e.g., varied word choices, action words, appropriate modifiers, sensory details)?\ndemonstrate control of a challenging vocabulary?\nDoes the writing sample demonstrate exceptional writing technique?\nIs the writing exceptionally fluent?\nDoes it include varied sentence patterns, including complex sentences?\nDoes it demonstrate use of writer’s techniques (e.g., literary conventions such as imagery and dialogue and/or literary genres such as humor and suspense)?\nVoice\nDoes the writing sample demonstrate effective adjustment of language and tone to task and reader? Does it\nexhibit appropriate register (e.g., formal, personal, or dialect) to suit task?\ndemonstrate a strong sense of audience?\nexhibit an original perspective (e.g., authoritative, lively, and/or exciting)?\nScore Point 5: A Score Point 5 paper represents a solid performance. It fully accomplishes the task, but lacks the overall level of sophistication and consistency of a Score Point 6 paper.\nIdeas and Content\nDoes the writing sample fully accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it\npresent a unifying theme or main idea without going off on tangents?\nstay focused on topic and task?\nDoes the writing sample include many relevant ideas? Does it\nprovide in-depth information and more than adequate supporting details that are developed?\nexplore many facets of the topic?\nOrganization\nAre the ideas in the writing sample organized logically? Does the writing\npresent a meaningful, cohesive whole with a beginning, a middle, and an end (i.e., include a solid introduction and conclusion)?\nprogress in an order that enhances meaning of text?\ninclude smooth transitions (e.g., use topic sentences) between sentences and paragraphs to enhance meaning of text? (Writing may have an occasional lapse.)\nStyle\nDoes the writing sample exhibit very good word usage? Does it\ninclude vocabulary to make explanations detailed and precise, descriptions rich, and actions clear and vivid?\ndemonstrate control of vocabulary?\nDoes the writing sample demonstrate very good writing technique?\nIs the writing very fluent?\nDoes it include varied sentence patterns, including complex sentences?\nDoes it demonstrate use of writer’s techniques (e.g., literary conventions such as imagery and dialogue and/or literary genres such as humor and suspense)?\nVoice\nDoes the writing sample demonstrate effective adjustment of language and tone to task and reader? Does it\nexhibit appropriate register (e.g., formal, personal, or dialect) to suit task?\ndemonstrate a sense of audience?\nexhibit an original perspective (e.g., authoritative, lively, and/or exciting)?\nScore Point 4: A Score Point 4 paper represents a good performance. It accomplishes the task, but generally needs to exhibit more development, better organization, or a more sophisticated writing style to receive a higher score.\nIdeas and Content\nDoes the writing sample accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it\npresent a unifying theme or main idea? (Writing may include minor tangents.)\nstay mostly focused on topic and task?\nDoes the writing sample include relevant ideas? Does it\ninclude sufficient information and supporting details? (Details may not be fully developed; ideas may be listed.)\nexplore some facets of the topic?\nOrganization\nAre the ideas in the writing sample organized logically? Does the writing\npresent a meaningful whole with a beginning, a middle, and an end despite an occasional lapse (e.g., a weak introduction or conclusion)?\ngenerally progress in an order that enhances meaning of text?\ninclude transitions between sentences and paragraphs to enhance meaning of text? (Transitions may be rough, although some topic sentences are included.)\nStyle\nDoes the writing sample exhibit good word usage? Does it\ninclude vocabulary that is appropriately chosen, with words that clearly convey the writer’s meaning?\ndemonstrate control of basic vocabulary?\nDoes the writing sample demonstrate good writing technique?\nIs the writing fluent?\nDoes it exhibit some varied sentence patterns, including some complex sentences?\nDoes it demonstrate an attempt to use writer’s techniques (e.g., literary conventions such as imagery and dialogue and/or literary genres such as humor and suspense)?\nVoice\nDoes the writing sample demonstrate an attempt to adjust language and tone to task and reader? Does it\ngenerally exhibit appropriate register (e.g., formal, personal, or dialect) to suit task? (The writing may occasionally slip out of register.)\ndemonstrate some sense of audience?\nattempt an original perspective?\nScore Point 3: A Score Point 3 paper represents a performance that minimally accomplishes the task. Some elements of development, organization, and writing style are weak.\nIdeas and Content\nDoes the writing sample minimally accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it\nattempt a unifying theme or main idea?\nstay somewhat focused on topic and task?\nDoes the writing sample include some relevant ideas? Does it\ninclude some information with only a few details, or list ideas without supporting details?\nexplore some facets of the topic?\nOrganization\nIs there an attempt to logically organize ideas in the writing sample? Does the writing\nhave a beginning, a middle, or an end that may be weak or absent?\ndemonstrate an attempt to progress in an order that enhances meaning? (Progression of text may sometimes be unclear or out of order.)\ndemonstrate an attempt to include transitions? (Are some topic sentences used? Are transitions between sentences and paragraphs weak or absent?)\nStyle\nDoes the writing sample exhibit ordinary word usage? Does it\ncontain basic vocabulary, with words that are predictable and common?\ndemonstrate some control of vocabulary?\nDoes the writing sample demonstrate average writing technique?\nIs the writing generally fluent?\nDoes it contain mostly simple sentences (although there may be an attempt at more varied sentence patterns)?\nIs it generally ordinary and predictable?\nVoice\nDoes the writing sample demonstrate an attempt to adjust language and tone to task and reader? Does it\ndemonstrate a difficulty in establishing a register (e.g., formal, personal, or dialect)?\ndemonstrate little sense of audience?\ngenerally lack an original perspective?\nScore Point 2: A Score Point 2 paper represents a performance that only partially accomplishes the task. Some responses may exhibit difficulty maintaining a focus. Others may be too brief to provide sufficient development of the topic or evidence of adequate organizational or writing style.\nIdeas and Content\nDoes the writing sample only partially accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Does it\nattempt a main idea?\nsometimes lose focus or ineffectively display focus?\nDoes the writing sample include few relevant ideas? Does it\ninclude little information and few or no details?\nexplore only one or two facets of the topic?\nOrganization\nIs there a minimal attempt to logically organize ideas in the writing sample?\nDoes the writing have only one or two of the three elements: beginning, middle, and end?\nIs the writing sometimes difficult to follow? (Progression of text may be confusing or unclear.)\nAre transitions weak or absent (e.g., few or no topic sentences)?\nStyle\nDoes the writing sample exhibit minimal word usage? Does it\ncontain limited vocabulary? (Some words may be used incorrectly.)\ndemonstrate minimal control of vocabulary?\nDoes the writing sample demonstrate minimal writing technique?\nDoes the writing exhibit some fluency?\nDoes it rely mostly on simple sentences?\nIs it often repetitive, predictable, or dull?\nVoice\nDoes the writing sample demonstrate language and tone that may be inappropriate to task and reader? Does it\ndemonstrate use of a register inappropriate to the task (e.g., slang or dialect in a formal setting)?\ndemonstrate little or no sense of audience?\nlack an original perspective?\nScore Point 1: A Score Point 1 paper represents a performance that fails to accomplish the task. It exhibits considerable difficulty in areas of development, organization, and writing style. The writing is generally either very brief or rambling and repetitive, sometimes resulting in a response that may be difficult to read or comprehend.\nIdeas and Content\nDoes the writing sample fail to accomplish the task (e.g., support an opinion, summarize, tell a story, or write an article)? Is it\ndifficult for the reader to discern the main idea?\ntoo brief or too repetitive to establish or maintain a focus?\nDoes the writing sample include very few relevant ideas?\nDoes it include little information with few or no details or unrelated details?\nIs it unsuccessful in attempts to explore any facets of the prompt?\nOrganization\nAre the ideas in the writing sample organized illogically?\nDoes it have only one or two of the three elements: beginning, middle, or end?\nIs it difficult to follow, with the order possibly difficult to discern?\nAre transitions weak or absent (e.g., without topic sentences)?\nStyle\nDoes the writing sample exhibit less than minimal word usage? Does it\ncontain limited vocabulary, with many words used incorrectly?\ndemonstrate minimal or less than minimal control of vocabulary?\nDoes the writing sample demonstrate less than minimal writing technique? Does it\nlack fluency?\ndemonstrate problems with sentence patterns?\nconsist of writing that is flat and lifeless?\nVoice\nDoes the writing sample demonstrate language and tone that may be inappropriate to task and reader? Does it\ndemonstrate difficulty in choosing an appropriate register?\ndemonstrate a lack of a sense of audience?\nlack an original perspective?\n\nScoring note: This notebook follows the LCES paper ASAP Prompt 2 setup and predicts Domain 1 / Writing Applications only, with score range 1-6. Domain 2 is not used for QWK in this run."
  },
  "3": {
    "prompt": "Source Essay\nROUGH ROAD AHEAD: Do Not Exceed Posted Speed Limit\nby Joe Kurmaskie\nFORGET THAT OLD SAYING ABOUT NEVER taking candy from strangers. No, a better piece of advice for the solo cyclist would be, “Never accept travel advice from a collection of old-timers who haven’t left the confines of their porches since Carter was in office.” It’s not that a group of old guys doesn’t know the terrain. With age comes wisdom and all that, but the world is a fluid place. Things change.\nAt a reservoir campground outside of Lodi, California, I enjoyed the serenity of an early-summer evening and some lively conversation with these old codgers. What I shouldn’t have done was let them have a peek at my map. Like a foolish youth, the next morning I followed their advice and launched out at first light along a “shortcut” that was to slice away hours from my ride to Yosemite National Park.\nThey’d sounded so sure of themselves when pointing out landmarks and spouting off towns I would come to along this breezy jaunt. Things began well enough. I rode into the morning with strong legs and a smile on my face. About forty miles into the pedal, I arrived at the first “town.” This place might have been a thriving little spot at one time—say, before the last world war—but on that morning it fit the traditional definition of a ghost town. I chuckled, checked my water supply, and moved on. The sun was beginning to beat down, but I barely noticed it. The cool pines and rushing rivers of Yosemite had my name written all over them.\nTwenty miles up the road, I came to a fork of sorts. One ramshackle shed, several rusty pumps, and a corral that couldn’t hold in the lamest mule greeted me. This sight was troubling. I had been hitting my water bottles pretty regularly, and I was traveling through the high deserts of California in June.\nI got down on my hands and knees, working the handle of the rusted water pump with all my strength. A tarlike substance oozed out, followed by brackish water feeling somewhere in the neighborhood of two hundred degrees. I pumped that handle for several minutes, but the water wouldn’t cool down. It didn’t matter. When I tried a drop or two, it had the flavor of battery acid.\nThe old guys had sworn the next town was only eighteen miles down the road. I could make that! I would conserve my water and go inward for an hour or so—a test of my inner spirit.\nNot two miles into this next section of the ride, I noticed the terrain changing. Flat road was replaced by short, rolling hills. After I had crested the first few of these, a large highway sign jumped out at me. It read: ROUGH ROAD AHEAD: DO NOT EXCEED POSTED SPEED LIMIT.\nThe speed limit was 55 mph. I was doing a water-depleting 12 mph. Sometimes life can feel so cruel.\nI toiled on. At some point, tumbleweeds crossed my path and a ridiculously large snake—it really did look like a diamondback—blocked the majority of the pavement in front of me. I eased past, trying to keep my balance in my dehydrated state.\nThe water bottles contained only a few tantalizing sips. Wide rings of dried sweat circled my shirt, and the growing realization that I could drop from heatstroke on a gorgeous day in June simply because I listened to some gentlemen who hadn’t been off their porch in decades, caused me to laugh.\nIt was a sad, hopeless laugh, mind you, but at least I still had the energy to feel sorry for myself. There was no one in sight, not a building, car, or structure of any kind. I began breaking the ride down into distances I could see on the horizon, telling myself that if I could make it that far, I’d be fi ne.\nOver one long, crippling hill, a building came into view. I wiped the sweat from my eyes to make sure it wasn’t a mirage, and tried not to get too excited. With what I believed was my last burst of energy, I maneuvered down the hill.\nIn an ironic twist that should please all sadists reading this, the building—abandoned years earlier, by the looks of it—had been a Welch’s Grape Juice factory and bottling plant. A sandblasted picture of a young boy pouring a refreshing glass of juice into his mouth could still be seen.\nI hung my head.\nThat smoky blues tune “Summertime” rattled around in the dry honeycombs of my deteriorating brain.\nI got back on the bike, but not before I gathered up a few pebbles and stuck them in my mouth. I’d read once that sucking on stones helps take your mind off thirst by allowing what spit you have left to circulate. With any luck I’d hit a bump and lodge one in my throat.\nIt didn’t really matter. I was going to die and the birds would pick me clean, leaving only some expensive outdoor gear and a diary with the last entry in praise of old men, their wisdom, and their keen sense of direction. I made a mental note to change that paragraph if it looked like I was going to lose consciousness for the last time.\nSomehow, I climbed away from the abandoned factory of juices and dreams, slowly gaining elevation while losing hope. Then, as easily as rounding a bend, my troubles, thirst, and fear were all behind me.\nGARY AND WILBER’S FISH CAMP—IF YOU WANT BAIT FOR THE BIG ONES, WE’RE YOUR BEST BET!\n“And the only bet,” I remember thinking.\nAs I stumbled into a rather modern bathroom and drank deeply from the sink, I had an overwhelming urge to seek out Gary and Wilber, kiss them, and buy some bait—any bait, even though I didn’t own a rod or reel.\nAn old guy sitting in a chair under some shade nodded in my direction. Cool water dripped from my head as I slumped against the wall beside him.\n“Where you headed in such a hurry?”\n“Yosemite,” I whispered.\n“Know the best way to get there?”\nI watched him from the corner of my eye for a long moment. He was even older than the group I’d listened to in Lodi.\n“Yes, sir! I own a very good map.”\nAnd I promised myself right then that I’d always stick to it in the future.\n“Rough Road Ahead” by Joe Kurmaskie, from Metal Cowboy, copyright © 1999 Joe Kurmaskie.\n\nPrompt\nWrite a response that explains how the features of the setting affect the cyclist. In your response, include examples from the essay that support your conclusion.",
    "rubric": "Rubric Guidelines\nScore 3: The response demonstrates an understanding of the complexities of the text.\nAddresses the demands of the question\nUses expressed and implied information from the text\nClarifies and extends understanding beyond the literal\nScore 2: The response demonstrates a partial or literal understanding of the text.\nAddresses the demands of the question, although may not develop all parts equally\nUses some expressed or implied information from the text to demonstrate understanding\nMay not fully connect the support to a conclusion or assertion made about the text(s)\nScore 1: The response shows evidence of a minimal understanding of the text.\nMay show evidence that some meaning has been derived from the text\nMay indicate a misreading of the text or the question\nMay lack information or explanation to support an understanding of the text in relation to the question\nScore 0: The response is completely irrelevant or incorrect, or there is no response."
  },
  "4": {
    "prompt": "Source Essay\nWinter Hibiscus by Minfong Ho\nSaeng, a teenage girl, and her family have moved to the United States from Vietnam. As Saeng walks home after failing her driver’s test, she sees a familiar plant. Later, she goes to a florist shop to see if the plant can be purchased.\nIt was like walking into another world. A hot, moist world exploding with greenery. Huge flat leaves, delicate wisps of tendrils, ferns and fronds and vines of all shades and shapes grew in seemingly random profusion.\n“Over there, in the corner, the hibiscus. Is that what you mean?” The florist pointed at a leafy potted plant by the corner.\nThere, in a shaft of the wan afternoon sunlight, was a single blood-red blossom, its five petals splayed back to reveal a long stamen tipped with yellow pollen. Saeng felt a shock of recognition so intense, it was almost visceral.1\n“Saebba,” Saeng whispered.\nA saebba hedge, tall and lush, had surrounded their garden, its lush green leaves dotted with vermilion flowers. And sometimes after a monsoon rain, a blossom or two would have blown into the well, so that when she drew the well water, she would find a red blossom floating in the bucket.\nSlowly, Saeng walked down the narrow aisle toward the hibiscus. Orchids, lanna bushes, oleanders, elephant ear begonias, and bougainvillea vines surrounded her. Plants that she had not even realized she had known but had forgotten drew her back into her childhood world.\nWhen she got to the hibiscus, she reached out and touched a petal gently. It felt smooth and cool, with a hint of velvet toward the center—just as she had known it would feel.\nAnd beside it was yet another old friend, a small shrub with waxy leaves and dainty flowers with purplish petals and white centers. “Madagascar periwinkle,” its tag announced. How strange to see it in a pot, Saeng thought. Back home it just grew wild, jutting out from the cracks in brick walls or between tiled roofs.\nAnd that rich, sweet scent—that was familiar, too. Saeng scanned the greenery around her and found a tall, gangly plant with exquisite little white blossoms on it.  “Dok Malik,” she said, savoring the feel of the word on her tongue, even as she silently noted the English name on its tag, “jasmine.”\nOne of the blossoms had fallen off, and carefully Saeng picked it up and smelled it. She closed her eyes and breathed in, deeply. The familiar fragrance filled her lungs, and Saeng could almost feel the light strands of her grandmother’s long gray hair, freshly washed, as she combed it out with the fine-toothed buffalo-horn comb. And when the sun had dried it, Saeng would help the gnarled old fingers knot the hair into a bun, then slip a dok Malik bud into it.\nSaeng looked at the white bud in her hand now, small and fragile. Gently, she closed her palm around it and held it tight. That, at least, she could hold on to. But where was the fine-toothed comb? The hibiscus hedge? The well? Her gentle grandmother?\nA wave of loss so deep and strong that it stung Saeng’s eyes now swept over her. A blink, a channel switch, a boat ride into the night, and it was all gone. Irretrievably, irrevocably gone.\nAnd in the warm moist shelter of the greenhouse, Saeng broke down and wept.\nIt was already dusk when Saeng reached home. The wind was blowing harder, tearing off the last remnants of green in the chicory weeds that were growing out of the cracks in the sidewalk. As if oblivious to the cold, her mother was still out in the vegetable garden, digging up the last of the onions with a rusty trowel. She did not see Saeng until the girl had quietly knelt down next to her.\nHer smile of welcome warmed Saeng. “Ghup ma laio le? You’re back?” she said cheerfully. “Goodness, it’s past five. What took you so long? How did it go? Did you—?” Then she noticed the potted plant that Saeng was holding, its leaves quivering in the wind.\nMrs. Panouvong uttered a small cry of surprise and delight. “Dok faeng-noi!” she said. “Where did you get it?”\n“I bought it,” Saeng answered, dreading her mother’s next question.\n“How much?”\nFor answer Saeng handed her mother some coins.\n“That’s all?” Mrs. Panouvong said, appalled, “Oh, but I forgot! You and the\nLambert boy ate Bee-Maags . . . .”\n“No, we didn’t, Mother,” Saeng said.\n“Then what else—?”\n“Nothing else. I paid over nineteen dollars for it.”\n“You what?” Her mother stared at her incredulously. “But how could you? All the seeds for this vegetable garden didn’t cost that much! You know how much we—” She paused, as she noticed the tearstains on her daughter’s cheeks and her puffy eyes.\n“What happened?” she asked, more gently.\n“I—I failed the test,” Saeng said.\nFor a long moment Mrs. Panouvong said nothing. Saeng did not dare look her mother in the eye. Instead, she stared at the hibiscus plant and nervously tore off a leaf, shredding it to bits.\nHer mother reached out and brushed the fragments of green off Saeng’s hands. “It’s a beautiful plant, this dok faeng-noi,” she finally said. “I’m glad you got it.”\n“It’s—it’s not a real one,” Saeng mumbled.\n“I mean, not like the kind we had at—at—” She found that she was still too shaky to say the words at home, lest she burst into tears again. “Not like the kind we had before,” she said.\n“I know,” her mother said quietly. “I’ve seen this kind blooming along the lake. Its flowers aren’t as pretty, but it’s strong enough to make it through the cold months here, this winter hibiscus. That’s what matters.”\nShe tipped the pot and deftly eased the ball of soil out, balancing the rest of the plant in her other hand. “Look how root-bound it is, poor thing,” she said. “Let’s plant it, right now.”\nShe went over to the corner of the vegetable patch and started to dig a hole in the ground. The soil was cold and hard, and she had trouble thrusting the shovel into it. Wisps of her gray hair trailed out in the breeze, and her slight frown deepened the wrinkles around her eyes. There was a frail, wiry beauty to her that touched Saeng deeply.\n“Here, let me help, Mother,” she offered, getting up and taking the shovel away from her.\nMrs. Panouvong made no resistance. “I’ll bring in the hot peppers and bitter melons, then, and start dinner. How would you like an omelet with slices of the bitter melon?”\n“I’d love it,” Saeng said.\nLeft alone in the garden, Saeng dug out a hole and carefully lowered the “winter hibiscus” into it. She could hear the sounds of cooking from the kitchen now, the beating of eggs against a bowl, the sizzle of hot oil in the pan. The pungent smell of bitter melon wafted out, and Saeng’s mouth watered. It was a cultivated taste, she had discovered—none of her classmates or friends, not even Mrs. Lambert, liked it—this sharp, bitter melon that left a golden aftertaste on the tongue. But she had grown up eating it and, she admitted to herself, much preferred it to a Big Mac.\nThe “winter hibiscus” was in the ground now, and Saeng tamped down the soil around it. Overhead, a flock of Canada geese flew by, their faint honks clear and—yes—familiar to Saeng now. Almost reluctantly, she realized that many of the things that she had thought of as strange before had become, through the quiet repetition of season upon season, almost familiar to her now. Like the geese. She lifted her head and watched as their distinctive V was etched against the evening sky, slowly fading into the distance.\nWhen they come back, Saeng vowed silently to herself, in the spring, when the snows melt and the geese return and this hibiscus is budding, then I will take that test again.\n“Winter Hibiscus” by Minfong Ho, copyright © 1993 by Minfong Ho, from Join In, Multiethnic Short Stories, by Donald R. Gallo, ed.\n\nPrompt\nRead the last paragraph of the story.\n\"When they come back, Saeng vowed silently to herself, in the spring, when the snows melt and the geese return and this hibiscus is budding, then I will take that test again.\"\nWrite a response that explains why the author concludes the story with this paragraph. In your response, include details and examples from the story that support your ideas.",
    "rubric": "Rubric Guidelines\nScore 3: The response demonstrates an understanding of the complexities of the text.\nAddresses the demands of the question\nUses expressed and implied information from the text\nClarifies and extends understanding beyond the literal\nScore 2: The response demonstrates a partial or literal understanding of the text.\nAddresses the demands of the question, although may not develop all parts equally\nUses some expressed or implied information from the text to demonstrate understanding\nMay not fully connect the support to a conclusion or assertion made about the text(s)\nScore 1: The response shows evidence of a minimal understanding of the text.\nMay show evidence that some meaning has been derived from the text\nMay indicate a misreading of the text or the question\nMay lack information or explanation to support an understanding of the text in relation to the question\nScore 0: The response is completely irrelevant or incorrect, or there is no response."
  },
  "5": {
    "prompt": "Source Essay\nNarciso Rodriguez\nfrom Home: The Blueprints of Our Lives\nMy parents, originally from Cuba, arrived in the United States in 1956. After living for a year in a furnished one-room apartment, twenty-one-year-old Rawedia Maria and twenty-seven-year-old Narciso Rodriguez, Sr., could afford to move into a modest, three-room apartment I would soon call home.\nIn 1961, I was born into this simple house, situated in a two-family, blond-brick building in the Ironbound section of Newark, New Jersey. Within its walls, my young parents created our traditional Cuban home, the very heart of which was the kitchen. My parents both shared cooking duties and unwittingly passed on to me their rich culinary skills and a love of cooking that is still with me today (and for which I am eternally grateful). Passionate Cuban music (which I adore to this day) filled the air, mixing with the aromas of the kitchen. Here, the innocence of childhood, the congregation of family and friends, and endless celebrations that encompassed both, formed the backdrop to life in our warm home.\nGrowing up in this environment instilled in me a great sense that “family” had nothing to do with being a blood relative. Quite the contrary, our neighborhood was made up of mostly Spanish, Cuban, and Italian immigrants at a time when overt racism was the norm and segregation prevailed in the United States. In our neighborhood, despite customs elsewhere, all of these cultures came together in great solidarity and friendship. It was a close-knit community of honest, hardworking immigrants who extended a hand to people who, while not necessarily their own kind, were clearly in need.\nOur landlord and his daughter, Alegria (my babysitter and first friend), lived above us, and Alegria graced our kitchen table for meals more often than not. Also at the table were Sergio and Edelmira, my surrogate grandparents who lived in the basement apartment. (I would not know my “real” grandparents, Narciso the Elder and Consuelo, until 1970 when they were allowed to leave Cuba.) My aunts Bertha and Juanita and my cousins Arnold, Maria, and Rosemary also all lived nearby and regularly joined us at our table. Countless extended family members came and went — and there was often someone staying with us temporarily until they were able to get back on their feet. My parents always kept their arms and their door open to the many people we considered family, knowing that they would do the same for us.\nMy mother and father had come to this country with such courage, without any knowledge of the language or the culture. They came selflessly, as many immigrants do, to give their children a better life, even though it meant leaving behind their families, friends, and careers in the country they loved. They struggled both personally and financially, braving the harsh northern winters while yearning for their native tropics and facing cultural hardships. The barriers to work were strong and high, and my parents both had to accept that they might not be able to find the kind of jobs they deserved. In Cuba, Narciso, Sr., had worked in a laboratory and Rawedia Maria had studied chemical engineering. In the United States, they had to start their lives over entirely, taking whatever work they could find. The faith that this struggle would lead them and their children to better times drove them to endure these hard times.\nI will always be grateful to my parents for their love and sacrifice. I’ve often told them that what they did was a much more courageous thing than I could have ever done. I’ve often told them of my admiration for their strength and perseverance, and I’ve thanked them repeatedly. But, in reality, there is no way to express my gratitude for the spirit of generosity impressed upon me at such an early age and the demonstration of how important family and friends are. These are two lessons that my parents did not just tell me. They showed me with their lives, and these teachings have been the basis of my life.\nIt was in this simple house that my parents welcomed other refugees to celebrate their arrival to this country and where I celebrated my first birthdays. It was in the warmth of the kitchen in this humble house where a Cuban feast (albeit a frugal Cuban feast) always filled the air with not just scent and music but life and love. It was here where I learned the real definition of “family.” And for this, I will never forget that house or its gracious neighborhood or the many things I learned there about how to love. I will never forget how my parents turned this simple house into a home.\n— Narciso Rodriguez, Fashion designer\nHometown: Newark, New Jersey\n“Narciso Rodriguez” by Narciso Rodriguez, from Home: The Blueprints of Our Lives. Copyright © 2006 by John Edwards.\n\nPrompt\nDescribe the mood created by the author in the memoir. Support your answer with relevant and specific information from the memoir.",
    "rubric": "Rubric Guidelines\nScore Point 4: The response is a clear, complete, and accurate description of the mood created by the author. The response includes relevant and specific information from the memoir.\nScore Point 3: The response is a mostly clear, complete, and accurate description of the mood created by the author. The response includes relevant but often general information from the memoir.\nScore Point 2: The response is a partial description of the mood created by the author. The response includes limited information from the memoir and may include misinterpretations.\nScore Point 1: The response is a minimal description of the mood created by the author. The response includes little or no information from the memoir and may include misinterpretations.\nOR\nThe response relates minimally to the task.\nScore Point 0: The response is incorrect or irrelevant or contains insufficient information to demonstrate comprehension."
  },
  "6": {
    "prompt": "Source Essay\nThe Mooring Mast\nby Marcia Amidon Lüsted\nWhen the Empire State Building was conceived, it was planned as the world’s tallest building, taller even than the new Chrysler Building that was being constructed at Forty-second Street and Lexington Avenue in New York. At seventy-seven stories, it was the tallest building before the Empire State began construction, and Al Smith was determined to outstrip it in height.\nThe architect building the Chrysler Building, however, had a trick up his sleeve. He secretly constructed a 185-foot spire inside the building, and then shocked the public and the media by hoisting it up to the top of the Chrysler Building, bringing it to a height of 1,046 feet, 46 feet taller than the originally announced height of the Empire State Building.\nAl Smith realized that he was close to losing the title of world’s tallest building, and on December 11, 1929, he announced that the Empire State would now reach the height of 1,250 feet. He would add a top or a hat to the building that would be even more distinctive than any other building in the city. John Tauranac describes the plan:\n[The top of the Empire State Building] would be more than ornamental, more than a spire or dome or a pyramid put there to add a desired few feet to the height of the building or to mask something as mundane as a water tank. Their top, they said, would serve a higher calling. The Empire State Building would be equipped for an age of transportation that was then only the dream of aviation pioneers.\nThis dream of the aviation pioneers was travel by dirigible, or zeppelin, and the Empire State Building was going to have a mooring mast at its top for docking these new airships, which would accommodate passengers on already existing transatlantic routes and new routes that were yet to come.\nThe Age of Dirigibles\nBy the 1920s, dirigibles were being hailed as the transportation of the future. Also known today as blimps, dirigibles were actually enormous steel-framed balloons, with envelopes of cotton fabric filled with hydrogen and helium to make them lighter than air. Unlike a balloon, a dirigible could be maneuvered by the use of propellers and rudders, and passengers could ride in the gondola, or enclosed compartment, under the balloon.\nDirigibles had a top speed of eighty miles per hour, and they could cruise at seventy miles per hour for thousands of miles without needing refueling. Some were as long as one thousand feet, the same length as four blocks in New York City. The one obstacle to their expanded use in New York City was the lack of a suitable landing area. Al Smith saw an opportunity for his Empire State Building: A mooring mast added to the top of the building would allow dirigibles to anchor there for several hours for refueling or service, and to let passengers off and on. Dirigibles were docked by means of an electric winch, which hauled in a line from the front of the ship and then tied it to a mast. The body of the dirigible could swing in the breeze, and yet passengers could safely get on and off the dirigible by walking down a gangplank to an open observation platform.\nThe architects and engineers of the Empire State Building consulted with experts, taking tours of the equipment and mooring operations at the U.S. Naval Air Station in Lakehurst, New Jersey. The navy was the leader in the research and development of dirigibles in the United States. The navy even offered its dirigible, the Los Angeles, to be used in testing the mast. The architects also met with the president of a recently formed airship transport company that planned to offer dirigible service across the Pacific Ocean.\nWhen asked about the mooring mast, Al Smith commented:\n[It’s] on the level, all right. No kidding. We’re working on the thing now. One set of engineers here in New York is trying to dope out a practical, workable arrangement and the Government people in Washington are figuring on some safe way of mooring airships to this mast.\nDesigning the Mast\nThe architects could not simply drop a mooring mast on top of the Empire State Building’s flat roof. A thousand-foot dirigible moored at the top of the building, held by a single cable tether, would add stress to the building’s frame. The stress of the dirigible’s load and the wind pressure would have to be transmitted all the way to the building’s foundation, which was nearly eleven hundred feet below. The steel frame of the Empire State Building would have to be modified and strengthened to accommodate this new situation. Over sixty thousand dollars’ worth of modifications had to be made to the building’s framework.\nRather than building a utilitarian mast without any ornamentation, the architects designed a shiny glass and chrome-nickel stainless steel tower that would be illuminated from inside, with a stepped-back design that imitated the overall shape of the building itself. The rocket-shaped mast would have four wings at its corners, of shiny aluminum, and would rise to a conical roof that would house the mooring arm. The winches and control machinery for the dirigible mooring would be housed in the base of the shaft itself, which also housed elevators and stairs to bring passengers down to the eighty-sixth floor, where baggage and ticket areas would be located.\nThe building would now be 102 floors, with a glassed-in observation area on the 101st floor and an open observation platform on the 102nd floor. This observation area was to double as the boarding area for dirigible passengers.\nOnce the architects had designed the mooring mast and made changes to the existing plans for the building’s skeleton, construction proceeded as planned. When the building had been framed to the 85th floor, the roof had to be completed before the framing for the mooring mast could take place. The mast also had a skeleton of steel and was clad in stainless steel with glass windows. Two months after the workers celebrated framing the entire building, they were back to raise an American flag again—this time at the top of the frame for the mooring mast.\nThe Fate of the Mast\nThe mooring mast of the Empire State Building was destined to never fulfill its purpose, for reasons that should have been apparent before it was ever constructed. The greatest reason was one of safety: Most dirigibles from outside of the United States used hydrogen rather than helium, and hydrogen is highly flammable. When the German dirigible Hindenburg was destroyed by fire in Lakehurst, New Jersey, on May 6, 1937, the owners of the Empire State Building realized how much worse that accident could have been if it had taken place above a densely populated area such as downtown New York.\nThe greatest obstacle to the successful use of the mooring mast was nature itself. The winds on top of the building were constantly shifting due to violent air currents. Even if the dirigible were tethered to the mooring mast, the back of the ship would swivel around and around the mooring mast. Dirigibles moored in open landing fields could be weighted down in the back with lead weights, but using these at the Empire State Building, where they would be dangling high above pedestrians on the street, was neither practical nor safe.\nThe other practical reason why dirigibles could not moor at the Empire State Building was an existing law against airships flying too low over urban areas. This law would make it illegal for a ship to ever tie up to the building or even approach the area, although two dirigibles did attempt to reach the building before the entire idea was dropped. In December 1930, the U.S. Navy dirigible Los Angeles approached the mooring mast but could not get close enough to tie up because of forceful winds. Fearing that the wind would blow the dirigible onto the sharp spires of other buildings in the area, which would puncture the dirigible’s shell, the captain could not even take his hands off the control levers.\nTwo weeks later, another dirigible, the Goodyear blimp Columbia, attempted a publicity stunt where it would tie up and deliver a bundle of newspapers to the Empire State Building. Because the complete dirigible mooring equipment had never been installed, a worker atop the mooring mast would have to catch the bundle of papers on a rope dangling from the blimp. The papers were delivered in this fashion, but after this stunt the idea of using the mooring mast was shelved. In February 1931, Irving Clavan of the building’s architectural office said, “The as yet unsolved problems of mooring air ships to a fixed mast at such a height made it desirable to postpone to a later date the final installation of the landing gear.”\nBy the late 1930s, the idea of using the mooring mast for dirigibles and their passengers had quietly disappeared. Dirigibles, instead of becoming the transportation of the future, had given way to airplanes. The rooms in the Empire State Building that had been set aside for the ticketing and baggage of dirigible passengers were made over into the world’s highest soda fountain and tea garden for use by the sightseers who flocked to the observation decks. The highest open observation deck, intended for disembarking passengers, has never been open to the public.\n“The Mooring Mast” by Marcia Amidon Lüsted, from The Empire State Building. Copyright © 2004 by Gale, a part of Cengage Learning, Inc.\n\nPrompt\nBased on the excerpt, describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. Support your answer with relevant and specific information from the excerpt.",
    "rubric": "Rubric Guidelines\nScore Point 4: The response is a clear, complete, and accurate description of the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. The response includes relevant and specific information from the excerpt.\nScore Point 3: The response is a mostly clear, complete, and accurate description of the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. The response includes relevant but often general information from the excerpt.\nScore Point 2: The response is a partial description of the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. The response includes limited information from the excerpt and may include misinterpretations.\nScore Point 1: The response is a minimal description of the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. The response includes little or no information from the excerpt and may include misinterpretations.\nOR\nThe response relates minimally to the task.\nScore Point 0: The response is totally incorrect or irrelevant, or contains insufficient evidence to demonstrate comprehension."
  },
  "7": {
    "prompt": "Prompt\nWrite about patience. Being patient means that you are understanding and tolerant. A patient person experience difficulties without complaining.\nDo only one of the following: write a story about a time when you were patient OR write a story about a time when someone you know was patient OR write a story in your own way about patience.",
    "rubric": "Rubric Guidelines\nA rating of 0-3 on the following four traits:\nIdeas (points doubled)\nScore 3: Tells a story with ideas that are clearly focused on the topic and are thoroughly developed with specific, relevant details.\nScore 2: Tells a story with ideas that are somewhat focused on the topic and are developed with a mix of specific and/or general details.\nScore 1: Tells a story with ideas that are minimally focused on the topic and developed with limited and/or general details.\nScore 0: Ideas are not focused on the task and/or are undeveloped.\nOrganization\nScore 3: Organization and connections between ideas and/or events are clear and logically sequenced.\nScore 2: Organization and connections between ideas and/or events are logically sequenced.\nScore 1: Organization and connections between ideas and/or events are weak.\nScore 0: No organization evident.\nStyle\nScore 3: Command of language, including effective and compelling word choice and varied sentence structure, clearly supports the writer's purpose and audience.\nScore 2: Adequate command of language, including effective word choice and clear sentences, supports the writer's purpose and audience.\nScore 1: Limited use of language, including lack of variety in word choice and sentences, may hinder support for the writer's purpose and audience.\nScore 0: Ineffective use of language for the writer's purpose and audience.\nConventions\nScore 3: Consistent, appropriate use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.\nScore 2: Adequate use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.\nScore 1: Limited use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation for the grade level.\nScore 0: Ineffective use of conventions of Standard English for grammar, usage, spelling, capitalization, and punctuation.\n\nScoring note: The one-reader rubric totals 0-15. Ideas are doubled; Organization, Style, and Conventions are added once. The resolved score range used for evaluation is 0-30."
  },
  "8": {
    "prompt": "Prompt\nWe all understand the benefits of laughter. For example, someone once said, “Laughter is the shortest distance between two people.” Many other people believe that laughter is an important part of any relationship. Tell a true story in which laughter was one element or part.",
    "rubric": "Rubric Guidelines\nA rating of 1-6 on the following six traits:\nIdeas and Content\nScore 6: The writing is exceptionally clear, focused, and interesting. It holds the reader’s attention throughout. Main ideas stand out and are developed by strong support and rich details suitable to audience and purpose. The writing is characterized by\nclarity, focus, and control.\nmain idea(s) that stand out.\nsupporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.\na thorough, balanced, in-depth explanation / exploration of the topic; the writing makes connections and shares insights.\ncontent and selected details that are well-suited to audience and purpose.\nScore 5: The writing is clear, focused and interesting. It holds the reader’s attention. Main ideas stand out and are developed by supporting details suitable to audience and purpose. The writing is characterized by\nclarity, focus, and control.\nmain idea(s) that stand out.\nsupporting, relevant, carefully selected details; when appropriate, use of resources provides strong, accurate, credible support.\na thorough, balanced explanation / exploration of the topic; the writing makes connections and shares insights.\ncontent and selected details that are well-suited to audience and purpose.\nScore 4: The writing is clear and focused. The reader can easily understand the main ideas. Support is present, although it may be limited or rather general. The writing is characterized by\nan easily identifiable purpose.\nclear main idea(s).\nsupporting details that are relevant, but may be overly general or limited in places; when appropriate, resources are used to provide accurate support.\na topic that is explored / explained, although developmental details may occasionally be out of balance with the main idea(s); some connections and insights may be present.\ncontent and selected details that are relevant, but perhaps not consistently well-chosen for audience and purpose.\nScore 3: The reader can understand the main ideas, although they may be overly broad or simplistic, and the results may not be effective. Supporting detail is often limited, insubstantial, overly general, or occasionally slightly off-topic. The writing is characterized by\nan easily identifiable purpose and main idea(s).\npredictable or overly-obvious main ideas; or points that echo observations heard elsewhere; or a close retelling of another work.\nsupport that is attempted, but developmental details are often limited, uneven, somewhat off-topic, predictable, or too general (e.g., a list of underdeveloped points).\ndetails that may not be well-grounded in credible resources; they may be based on clichés, stereotypes or questionable sources of information.\ndifficulties when moving from general observations to specifics.\nScore 2: Main ideas and purpose are somewhat unclear or development is attempted but minimal. The writing is characterized by\na purpose and main idea(s) that may require extensive inferences by the reader.\nminimal development; insufficient details.\nirrelevant details that clutter the text.\nextensive repetition of detail.\nScore 1: The writing lacks a central idea or purpose. The writing is characterized by\nideas that are extremely limited or simply unclear.\nattempts at development that are minimal or nonexistent; the paper is too short to demonstrate the development of an idea.\nOrganization\nScore 6: The organization enhances the central idea(s) and its development. The order and structure are compelling and move the reader through the text easily. The writing is characterized by\neffective, perhaps creative, sequencing and paragraph breaks; the organizational structure fits the topic, and the writing is easy to follow.\na strong, inviting beginning that draws the reader in and a strong, satisfying sense of resolution or closure.\nsmooth, effective transitions among all elements (sentences, paragraphs, ideas).\ndetails that fit where placed.\nScore 5: The organization enhances the central idea(s) and its development. The order and structure are strong and move the reader through the text. The writing is characterized by\neffective sequencing and paragraph breaks; the organizational structure fits the topic, and the writing is easy to follow.\nan inviting beginning that draws the reader in and a satisfying sense of resolution or closure.\nsmooth, effective transitions among all elements (sentences, paragraphs, ideas).\ndetails that fit where placed.\nScore 4: Organization is clear and coherent. Order and structure are present, but may seem formulaic. The writing is characterized by\nclear sequencing and paragraph breaks.\nan organization that may be predictable.\na recognizable, developed beginning that may not be particularly inviting; a developed conclusion that may lack subtlety.\na body that is easy to follow with details that fit where placed.\ntransitions that may be stilted or formulaic.\norganization which helps the reader, despite some weaknesses.\nScore 3: An attempt has been made to organize the writing; however, the overall structure is inconsistent or skeletal. The writing is characterized by\nattempts at sequencing and paragraph breaks, but the order or the relationship among ideas may occasionally be unclear.\na beginning and an ending which, although present, are either undeveloped or too obvious (e.g., “My topic is...”; “These are all the reasons that...”).\ntransitions that sometimes work. The same few transitional devices (e.g., coordinating conjunctions, numbering, etc.) may be overused.\na structure that is skeletal or too rigid.\nplacement of details that may not always be effective.\norganization which lapses in some places, but helps the reader in others.\nScore 2: The writing lacks a clear organizational structure. An occasional organizational device is discernible; however, the writing is either difficult to follow and the reader has to reread substantial portions, or the piece is simply too short to demonstrate organizational skills. The writing is characterized by\nsome attempts at sequencing, but the order or the relationship among ideas is frequently unclear; a lack of paragraph breaks.\na missing or extremely undeveloped beginning, body, and/or ending.\na lack of transitions, or when present, ineffective or overused.\na lack of an effective organizational structure.\ndetails that seem to be randomly placed, leaving the reader frequently confused.\nScore 1: The writing lacks coherence; organization seems haphazard and disjointed. Even after rereading, the reader remains confused. The writing is characterized by\na lack of effective sequencing and paragraph breaks.\na failure to provide an identifiable beginning, body and/or ending.\na lack of transitions.\npacing that is consistently awkward; the reader feels either mired down in trivia or rushed along too rapidly.\na lack of organization which ultimately obscures or distorts the main point.\nVoice\nScore 6: The writer has chosen a voice appropriate for the topic, purpose, and audience. The writer demonstrates deep commitment to the topic, and there is an exceptional sense of “writing to be read.” The writing is expressive, engaging, or sincere. The writing is characterized by\nan effective level of closeness to or distance from the audience (e.g., a narrative should have a strong personal voice, while an expository piece may require extensive use of outside resources and a more academic voice; nevertheless, both should be engaging, lively, or interesting. Technical writing may require greater distance.).\nan exceptionally strong sense of audience; the writer seems to be aware of the reader and of how to communicate the message most effectively. The reader may discern the writer behind the words and feel a sense of interaction.\na sense that the topic has come to life; when appropriate, the writing may show originality, liveliness, honesty, conviction, excitement, humor, or suspense.\nScore 5: The writer has chosen a voice appropriate for the topic, purpose, and audience. The writer demonstrates commitment to the topic, and there is a sense of “writing to be read.” The writing is expressive, engaging, or sincere. The writing is characterized by an appropriate level of closeness to or distance from the audience (e.g., a narrative should have a strong personal voice, while an expository piece may require extensive use of outside resources and a more academic voice; nevertheless, both should be engaging, lively, or interesting. Technical writing may require greater distance.).\na strong sense of audience; the writer seems to be aware of the reader and of how to communicate the message most effectively. The reader may discern the writer behind the words and feel a sense of interaction.\na sense that the topic has come to life; when appropriate, the writing may show originality, liveliness, honesty, conviction, excitement, humor, or suspense.\nScore 4: A voice is present. The writer seems committed to the topic, and there may be a sense of “writing to be read.” In places, the writing is expressive, engaging, or sincere. The writing is characterized by\na suitable level of closeness to or distance from the audience.\na sense of audience; the writer seems to be aware of the reader but has not consistently employed an appropriate voice. The reader may glimpse the writer behind the words and feel a sense of interaction in places.\nliveliness, sincerity, or humor when appropriate; however, at times the writing may be either inappropriately casual or personal, or inappropriately formal and stiff.\nScore 3: The writer’s commitment to the topic seems inconsistent. A sense of the writer may emerge at times; however, the voice is either inappropriately personal or inappropriately impersonal. The writing is characterized by\na limited sense of audience; the writer’s awareness of the reader is unclear.\nan occasional sense of the writer behind the words; however, the voice may shift or disappear a line or two later and the writing become somewhat mechanical.\na limited ability to shift to a more objective voice when necessary.\ntext that is too short to demonstrate a consistent and appropriate voice.\nScore 2: The writing provides little sense of involvement or commitment. There is no evidence that the writer has chosen a suitable voice. The writing is characterized by\nlittle engagement of the writer; the writing tends to be largely flat, lifeless, stiff, or mechanical.\na voice that is likely to be overly informal and personal.\na lack of audience awareness; there is little sense of “writing to be read.”\nlittle or no hint of the writer behind the words. There is rarely a sense of interaction between reader and writer.\nScore 1: The writing seems to lack a sense of involvement or commitment. The writing is characterized by\nno engagement of the writer; the writing is flat and lifeless.\na lack of audience awareness; there is no sense of “writing to be read.”\nno hint of the writer behind the words. There is no sense of interaction between writer and reader; the writing does not involve or engage the reader.\nWord Choice\nScore 6: Words convey the intended message in an exceptionally interesting, precise, and natural way appropriate to audience and purpose. The writer employs a rich, broad range of words which have been carefully chosen and thoughtfully placed for impact. The writing is characterized by\naccurate, strong, specific words; powerful words energize the writing.\nfresh, original expression; slang, if used, seems purposeful and is effective.\nvocabulary that is striking and varied, but that is natural and not overdone.\nordinary words used in an unusual way.\nwords that evoke strong images; figurative language may be used.\nScore 5: Words convey the intended message in an interesting, precise, and natural way appropriate to audience and purpose. The writer employs a broad range of words which have been carefully chosen and thoughtfully placed for impact. The writing is characterized by\naccurate, specific words; word choices energize the writing.\nfresh, vivid expression; slang, if used, seems purposeful and is effective.\nvocabulary that may be striking and varied, but that is natural and not overdone.\nordinary words used in an unusual way.\nwords that evoke clear images; figurative language may be used.\nScore 4: Words effectively convey the intended message. The writer employs a variety of words that are functional and appropriate to audience and purpose. The writing is characterized by\nwords that work but do not particularly energize the writing.\nexpression that is functional; however, slang, if used, does not seem purposeful and is not particularly effective.\nattempts at colorful language that may occasionally seem overdone.\noccasional overuse of technical language or jargon.\nrare experiments with language; however, the writing may have some fine moments and generally avoids clichés.\nScore 3: Language lacks precision and variety, or may be inappropriate to audience and purpose in places. The writer does not employ a variety of words, producing a sort of “generic” paper filled with familiar words and phrases. The writing is characterized by\nwords that work, but that rarely capture the reader’s interest.\nexpression that seems mundane and general; slang, if used, does not seem purposeful and is not effective.\nattempts at colorful language that seem overdone or forced.\nwords that are accurate for the most part, although misused words may occasionally appear; technical language or jargon may be overused or inappropriately used.\nreliance on clichés and overused expressions.\ntext that is too short to demonstrate variety.\nScore 2: Language is monotonous and/or misused, detracting from the meaning and impact. The writing is characterized by\nwords that are colorless, flat or imprecise.\nmonotonous repetition or overwhelming reliance on worn expressions that repeatedly detract from the message.\nimages that are fuzzy or absent altogether.\nScore 1: The writing shows an extremely limited vocabulary or is so filled with misuses of words that the meaning is obscured. Only the most general kind of message is communicated because of vague or imprecise language. The writing is characterized by\ngeneral, vague words that fail to communicate.\nan extremely limited range of words.\nwords that simply do not fit the text; they seem imprecise, inadequate, or just plain wrong.\nSentence Fluency\nScore 6: The writing has an effective flow and rhythm. Sentences show a high degree of craftsmanship, with consistently strong and varied structure that makes expressive oral reading easy and enjoyable. The writing is characterized by\na natural, fluent sound; it glides along with one sentence flowing effortlessly into the next.\nextensive variation in sentence structure, length, and beginnings that add interest to the text.\nsentence structure that enhances meaning by drawing attention to key ideas or reinforcing relationships among ideas.\nvaried sentence patterns that create an effective combination of power and grace.\nstrong control over sentence structure; fragments, if used at all, work well.\nstylistic control; dialogue, if used, sounds natural.\nScore 5: The writing has an easy flow and rhythm. Sentences are carefully crafted, with strong and varied structure that makes expressive oral reading easy and enjoyable. The writing is characterized by\na natural, fluent sound; it glides along with one sentence flowing into the next.\nvariation in sentence structure, length, and beginnings that add interest to the text.\nsentence structure that enhances meaning.\ncontrol over sentence structure; fragments, if used at all, work well.\nstylistic control; dialogue, if used, sounds natural.\nScore 4: The writing flows; however, connections between phrases or sentences may be less than fluid. Sentence patterns are somewhat varied, contributing to ease in oral reading. The writing is characterized by\na natural sound; the reader can move easily through the piece, although it may lack a certain rhythm and grace.\nsome repeated patterns of sentence structure, length, and beginnings that may detract somewhat from overall impact.\nstrong control over simple sentence structures, but variable control over more complex sentences; fragments, if present, are usually effective.\noccasional lapses in stylistic control; dialogue, if used, sounds natural for the most part, but may at times sound stilted or unnatural.\nScore 3: The writing tends to be mechanical rather than fluid. Occasional awkward constructions may force the reader to slow down or reread. The writing is characterized by\nsome passages that invite fluid oral reading; however, others do not.\nsome variety in sentence structure, length, and beginnings, although the writer falls into repetitive sentence patterns.\ngood control over simple sentence structures, but little control over more complex sentences; fragments, if present, may not be effective.\nsentences which, although functional, lack energy.\nlapses in stylistic control; dialogue, if used, may sound stilted or unnatural.\ntext that is too short to demonstrate variety and control.\nScore 2: The writing tends to be either choppy or rambling. Awkward constructions often force the reader to slow down or reread. The writing is characterized by\nsignificant portions of the text that are difficult to follow or read aloud.\nsentence patterns that are monotonous (e.g., subject-verb or subject-verb-object).\na significant number of awkward, choppy, or rambling constructions.\nScore 1: The writing is difficult to follow or to read aloud. Sentences tend to be incomplete, rambling, or very awkward. The writing is characterized by\ntext that does not invite—and may not even permit—smooth oral reading.\nconfusing word order that is often jarring and irregular.\nsentence structure that frequently obscures meaning.\nsentences that are disjointed, confusing, or rambling.\nConventions\nScore 6: The writing demonstrates exceptionally strong control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage) and uses them effectively to enhance communication. Errors are so few and so minor that the reader can easily skim right over them unless specifically searching for them. The writing is characterized by\nstrong control of conventions; manipulation of conventions may occur for stylistic effect.\nstrong, effective use of punctuation that guides the reader through the text.\ncorrect spelling, even of more difficult words.\ncorrect grammar and usage that contribute to clarity and style.\nskill in using a wide range of conventions in a sufficiently long and complex piece.\nlittle or no need for editing.\nScore 5: The writing demonstrates strong control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage) and uses them effectively to enhance communication. Errors are few and minor. Conventions support readability. The writing is characterized by\nstrong control of conventions.\neffective use of punctuation that guides the reader through the text.\ncorrect spelling, even of more difficult words.\ncorrect capitalization; errors, if any, are minor.\ncorrect grammar and usage that contribute to clarity and style.\nskill in using a wide range of conventions in a sufficiently long and complex piece.\nlittle need for editing.\nScore 4: The writing demonstrates control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage). Significant errors do not occur frequently. Minor errors, while perhaps noticeable, do not impede readability. The writing is characterized by\ncontrol over conventions used, although a wide range is not demonstrated.\ncorrect end-of-sentence punctuation; internal punctuation may sometimes be incorrect.\nspelling that is usually correct, especially on common words.\ncorrect capitalization; errors, if any, are minor.\noccasional lapses in correct grammar and usage; problems are not severe enough to distort meaning or confuse the reader.\nmoderate need for editing.\nScore 3: The writing demonstrates limited control of standard writing conventions (e.g., punctuation, spelling, capitalization, grammar and usage). Errors begin to impede readability. The writing is characterized by\nsome control over basic conventions; the text may be too simple or too short to reveal mastery.\nend-of-sentence punctuation that is usually correct; however, internal punctuation contains frequent errors.\nspelling errors that distract the reader; misspelling of common words occurs.\ncapitalization errors.\nerrors in grammar and usage that do not block meaning but do distract the reader.\nsignificant need for editing.\nScore 2: The writing demonstrates little control of standard writing conventions. Frequent, significant errors impede readability. The writing is characterized by\nlittle control over basic conventions.\nmany end-of-sentence punctuation errors; internal punctuation contains frequent errors.\nspelling errors that frequently distract the reader; misspelling of common words often occurs.\ncapitalization that is inconsistent or often incorrect.\nerrors in grammar and usage that interfere with readability and meaning.\nsubstantial need for editing.\nScore 1: Numerous errors in usage, spelling, capitalization, and punctuation repeatedly distract the reader and make the text difficult to read. In fact, the severity and frequency of errors are so overwhelming that the reader finds it difficult to focus on the message and must reread for meaning. The writing is characterized by\nvery limited skill in using conventions.\nbasic punctuation (including end-of-sentence punctuation) that tends to be omitted, haphazard, or incorrect.\nfrequent spelling errors that significantly impair readability.\ncapitalization that appears to be random.\na need for extensive editing.\nAdjudication Rules\nEach student essay is rated for six Writing traits (I, O, V, W, S, C), by two independent raters: Rater 1 and Rater 2. Rater 3 provides a third (resolution) rating for each trait, triggered by the following rules:\nStandard Rule: Non-adjacency between the 1st and 2nd scorer on any of the 6 traits generates a resolution read.\nCusp Rule: If first or second score has all 4s on:\nIdeas and Content\nOrganization\nSentence Fluency\nConventions,\nand the other (1st or 2nd score) has one 3 and three 4s in these categories, require a resolution. Voice and Word Choice are excluded – it does not matter what scores occur for Voice or Word choice (though non-adjacent Voice and Word Choice scores will still cause failure on (1)).\nTotal Composite Score:\nFor most essays:\n= (I_R1+I_R2)  +  (O_R1+O_R2)  + (S_R1+S_R2)  +  2 (C_R1+C_R2)\nWhen there is Rater 3 set of scores for the essay then the Total Composite Score formula changes to:\n=  2 (I_R3) + 2 (O_R3) + 2 (S_R3) + 4 (C_R3)    or equivalently   2 (I+O+S+C) +  2 (C)\nNote the use of only four of the six traits.\n\nScoring note: For the final composite score in this notebook, compare essays primarily on the traits used in the Total Composite Score: Ideas and Content, Organization, Sentence Fluency, and Conventions, with Conventions double-weighted. Voice and Word Choice are present in the official rubric but are not part of the final composite formula."
  }
}''')
ASAP_TASKS["6"]["prompt"] = """
Prompt
The essay is based on the excerpt "The Mooring Mast," about the Empire State Building's planned dirigible docking mast.

Relevant source summary:
The Empire State Building was designed to become the world's tallest building and included a planned mooring mast at the top so dirigibles could dock, refuel, and load or unload passengers. The idea came from the belief that dirigibles might be the future of transportation.

The builders faced several obstacles:
- Structural stress: a huge dirigible attached by a cable would put major wind and load stress on the building frame, requiring expensive structural modifications.
- Docking equipment and passenger access: winches, control machinery, observation platforms, elevators, baggage areas, and ticketing spaces had to be designed into the top floors.
- Dangerous winds: air currents around the top of the building shifted constantly, making it hard and unsafe for a dirigible to approach or stay attached.
- Safety risks: hydrogen-filled dirigibles could catch fire, and an accident over crowded New York City would be disastrous.
- Practical/legal limits: laws restricted airships from flying too low over urban areas.
- Failed attempts: the Navy dirigible Los Angeles could not dock because of forceful winds, and the Goodyear blimp Columbia only delivered newspapers as a stunt.

Task:
Based on the excerpt, describe the obstacles the builders of the Empire State Building faced in attempting to allow dirigibles to dock there. Support your answer with relevant and specific information from the excerpt.
""".strip()

print("P6 compact prompt chars:", len(ASAP_TASKS["6"]["prompt"]))
print("P6 compact rubric chars:", len(ASAP_TASKS["6"]["rubric"]))
ASAP_TASKS["6"]["rubric"] = """
Rubric Guidelines — ASAP Prompt 6, score 0–4.

Evaluate how well the response describes the obstacles the builders of the Empire State Building faced in trying to allow dirigibles to dock there. The response should use relevant and specific information from the excerpt.

Score 4: Clear, complete, and accurate explanation of the major obstacles, with relevant and specific evidence from the text. Shows strong understanding of the excerpt.

Score 3: Mostly clear, complete, and accurate explanation. Includes relevant evidence, though support may be more general or less fully developed.

Score 2: Partial explanation. Mentions some obstacles but may be incomplete, vague, weakly supported, or include some misunderstanding.

Score 1: Minimal explanation. Shows little understanding, gives little or no relevant evidence, or relates only weakly to the task.

Score 0: Incorrect, irrelevant, blank, or too insufficient to show comprehension.

Prefer the essay more likely to receive the higher score. Choose "tie" only if both would likely receive the same score.
""".strip()


ASAP_TASKS["7"]["rubric"] = """
Rubric Guidelines — ASAP Prompt 7, final score range 0–30.

Evaluate the story about patience using four traits: Ideas, Organization, Style, and Conventions. Ideas are most important and are doubled in the final score.

Ideas: Higher scores tell a focused story about patience with clear, relevant, specific, and well-developed details. Lower scores are unfocused, undeveloped, off-task, or too general.

Organization: Higher scores have clear, logical sequencing and strong connections between events or ideas. Lower scores have weak, confusing, or missing organization.

Style: Higher scores use effective language, appropriate word choice, and varied sentence structure to support the purpose and audience. Lower scores use limited, repetitive, or ineffective language.

Conventions: Higher scores show consistent control of grammar, usage, spelling, capitalization, and punctuation. Lower scores contain frequent errors that distract or interfere with meaning.

Prefer the essay more likely to receive the higher final composite score. Choose "tie" only if both would likely receive the same score.
""".strip()


ASAP_TASKS["8"]["rubric"] = """
Rubric Guidelines — ASAP Prompt 8, final score range 0–60.

Evaluate the true story about laughter using the traits that determine the composite score: Ideas and Content, Organization, Sentence Fluency, and Conventions, with Conventions weighted heavily.

Ideas and Content: Higher scores are clear, focused, engaging, and well developed with relevant details about laughter. Lower scores are unclear, thin, repetitive, off-topic, or insufficiently developed.

Organization: Higher scores have effective sequencing, paragraphing, transitions, and a satisfying beginning and ending. Lower scores are hard to follow, poorly sequenced, or lack clear structure.

Sentence Fluency: Higher scores have natural flow, varied sentence patterns, and readable rhythm. Lower scores are choppy, rambling, awkward, or difficult to read.

Conventions: Higher scores show strong control of grammar, spelling, punctuation, capitalization, and usage. Lower scores contain frequent errors that distract or impede meaning.

Prefer the essay more likely to receive the higher final composite score. Choose "tie" only if both would likely receive the same score.
""".strip()


for sid in ["6", "7", "8"]:
    print(f"P{sid} compact rubric chars:", len(ASAP_TASKS[sid]["rubric"]))
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

# Essay-token budgets used only when the full prompt would exceed LLM_MAX_INPUT_TOKENS.
# The rubric and task prompt are always preserved; only student essays are trimmed.
ESSAY_TOKEN_BUDGET_BY_SET = {
    1: 900,
    2: 900,
    3: 650,
    4: 650,
    5: 650,
    6: 650,
    7: 750,
    8: 900,
}

summary_rows = []
for sid in range(1, 9):
    task = ASAP_TASKS[str(sid)]
    summary_rows.append({
        "essay_set": sid,
        "score_range": ASAP_SCORE_RANGES[sid],
        "prompt_chars": len(task["prompt"]),
        "rubric_chars": len(task["rubric"]),
    })

pd.DataFrame(summary_rows)


P6 compact prompt chars: 1567
P6 compact rubric chars: 1259
P6 compact rubric chars: 1063
P7 compact rubric chars: 1102
P8 compact rubric chars: 1140


,essay_set,score_range,prompt_chars,rubric_chars
0,1,"(2, 12)",698,1874
1,2,"(1, 6)",731,11315
2,3,"(0, 3)",6133,973
3,4,"(0, 3)",8064,973
4,5,"(0, 4)",4935,933
5,6,"(0, 4)",1567,1063
6,7,"(0, 30)",343,1102
7,8,"(0, 60)",281,1140


In [5]:

# ============================================================
# 4. Load ASAP data
# ============================================================
def load_asap_data() -> pd.DataFrame:
    split_paths = {
        "train": TRAIN_PATH,
        "val": VAL_PATH,
        "test": TEST_PATH,
    }

    if all(os.path.exists(p) for p in split_paths.values()):
        frames = []
        for split, path in split_paths.items():
            df = pd.read_csv(path)
            df["split"] = split
            frames.append(df)
        data = pd.concat(frames, axis=0, ignore_index=True)
        print("Loaded prepared split CSV files.")
    elif os.path.exists(ASAP_TSV_PATH):
        data = pd.read_csv(ASAP_TSV_PATH, sep="\t", encoding="latin1")
        data["split"] = "all"
        print("Loaded original Kaggle TSV file.")
    else:
        raise FileNotFoundError(
            "Cannot find ASAP data. Upload either:\n"
            f"  - {TRAIN_PATH}, {VAL_PATH}, {TEST_PATH}\n"
            f"or\n  - {ASAP_TSV_PATH}"
        )

    required = {"essay_id", "essay_set", "essay"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "domain1_score" not in data.columns:
        candidates = ["resolved_score", "Resolved_Score", "score", "Score"]
        found = None
        for c in candidates:
            if c in data.columns:
                found = c
                break
        if found is None:
            raise ValueError("Cannot find domain1_score or an equivalent score column.")
        data["domain1_score"] = data[found]
        print(f"Using {found} as domain1_score.")

    data = data.copy()
    data["essay_id"] = data["essay_id"].astype(int)
    data["essay_set"] = data["essay_set"].astype(int)
    data["essay"] = data["essay"].fillna("").astype(str)
    data["domain1_score"] = data["domain1_score"].astype(int)

    # The LCES paper reports ASAP Prompt 2 as score range 1-6, so this notebook evaluates P2 on domain1_score.
    # domain2_score, if present, is intentionally not used in the paper-like ASAP-8 summary.
    return data

asap_df = load_asap_data()
print(asap_df.shape)
print(asap_df[["essay_id", "essay_set", "split", "domain1_score", "essay"]].head())

counts = asap_df.groupby("essay_set").agg(
    n=("essay_id", "count"),
    min_score=("domain1_score", "min"),
    max_score=("domain1_score", "max"),
).reset_index()
counts


Loaded prepared split CSV files.
(12976, 6)
   essay_id  essay_set  split  domain1_score  \
0     14876          6  train              3   
1      9985          4  train              0   
2      3231          2  train              4   
3     21137          8  train             39   
4     17919          7  train             23   

                                               essay  
0  In the excerpt The Mooring Mast by Marcia Amid...  
1  The author concluded this sentence because he ...  
2  I can think of several books that, I would not...  
3  My best friend @PERSON2 turned thirteen on a b...  
4  A time that I was patient is when I broke my f...  


,essay_set,n,min_score,max_score
0,1,1783,2,12
1,2,1800,1,6
2,3,1726,0,3
3,4,1770,0,3
4,5,1805,0,4
5,6,1800,0,4
6,7,1569,2,24
7,8,723,10,60


In [6]:

# ============================================================
# 5. Metrics, sampling, and data preparation helpers
# ============================================================
def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")
    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(spearman) if not pd.isna(spearman) else np.nan,
    }


def make_eval_subset(data: pd.DataFrame, essay_set: int, frac: Optional[float], seed: int) -> pd.DataFrame:
    df = data[data["essay_set"] == essay_set].copy().reset_index(drop=True)
    if df.empty:
        raise ValueError(f"No rows for essay_set={essay_set}")

    y_min, y_max = ASAP_SCORE_RANGES[essay_set]
    # Keep rows within the expected paper score range.
    bad = df[(df["domain1_score"] < y_min) | (df["domain1_score"] > y_max)]
    if len(bad) > 0:
        print(f"Warning: essay_set={essay_set} has {len(bad)} rows outside expected score range {y_min}-{y_max}.")

    if frac is not None and frac < 1.0:
        df = df.sample(frac=frac, random_state=seed + essay_set).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    return df


def sample_pairs_ordered(df: pd.DataFrame, num_pairs: int, seed: int) -> pd.DataFrame:
    """Sample unique ordered pairs (i, j), i != j, following the LCES definition."""
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).to_numpy()
    n = len(essay_ids)
    n_possible = n * (n - 1)
    if n < 2:
        raise ValueError("Need at least two essays to sample pairs.")

    target = min(int(num_pairs), int(n_possible))
    if target < num_pairs:
        print(f"Requested M={num_pairs}, but only {n_possible} ordered pairs exist. Using M={target}.")

    # If target is close to all possible pairs, enumerate to avoid rejection-sampling slowdown.
    if target > 0.20 * n_possible:
        all_pairs = np.array([(int(a), int(b)) for a in essay_ids for b in essay_ids if a != b], dtype=np.int64)
        chosen = rng.choice(len(all_pairs), size=target, replace=False)
        pairs = all_pairs[chosen]
    else:
        pairs_set = set()
        while len(pairs_set) < target:
            a, b = rng.choice(essay_ids, size=2, replace=False)
            pairs_set.add((int(a), int(b)))
        pairs = np.array(list(pairs_set), dtype=np.int64)
        rng.shuffle(pairs)

    return pd.DataFrame(pairs, columns=["essay_id_1", "essay_id_2"])


def prepare_all_subsets_and_pairs(data: pd.DataFrame) -> Dict[int, Dict[str, Any]]:
    prepared = {}
    for essay_set in RUN_ESSAY_SETS:
        subset_path = OUTPUT_DIR / "subsets" / f"p{essay_set}_eval_subset_seed{SEED}_frac{str(SAMPLE_FRAC).replace('.', '')}.csv"
        pairs_path = OUTPUT_DIR / "pairs" / f"p{essay_set}_pairs_m{M_PAIRWISE}_seed{SEED}.csv"

        if subset_path.exists():
            df_set = pd.read_csv(subset_path)
        else:
            df_set = make_eval_subset(data, essay_set, SAMPLE_FRAC, SEED)
            df_set.to_csv(subset_path, index=False)

        if pairs_path.exists():
            pairs = pd.read_csv(pairs_path)
        else:
            pairs = sample_pairs_ordered(df_set, M_PAIRWISE, seed=SEED + 1000 * essay_set)
            pairs.to_csv(pairs_path, index=False)

        used_ids = set(pairs["essay_id_1"].astype(int)) | set(pairs["essay_id_2"].astype(int))
        prepared[essay_set] = {"df": df_set, "pairs": pairs, "subset_path": subset_path, "pairs_path": pairs_path}
        print(
            f"P{essay_set}: essays={len(df_set):4d}, pairs={len(pairs):5d}, "
            f"coverage={len(used_ids) / len(df_set):.3f}, score_range={ASAP_SCORE_RANGES[essay_set]}"
        )
    return prepared

prepared_sets = prepare_all_subsets_and_pairs(asap_df)


P5: essays= 180, pairs= 5000, coverage=1.000, score_range=(0, 4)
P6: essays= 180, pairs= 5000, coverage=1.000, score_range=(0, 4)
P7: essays= 157, pairs= 5000, coverage=1.000, score_range=(0, 30)
P8: essays=  72, pairs= 5000, coverage=1.000, score_range=(0, 60)


In [7]:

# ============================================================
# 6. Load local pairwise LLM
# ============================================================
def load_local_llm(model_id: str):
    print("Loading tokenizer:", model_id)
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print("Loading model:", model_id)
    if LOAD_IN_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=True,
        )
    else:
        if torch.cuda.is_available():
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        else:
            dtype = torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=dtype,
            trust_remote_code=True,
        )

    model.config.pad_token_id = tok.pad_token_id
    model.generation_config.pad_token_id = tok.pad_token_id
    model.eval()
    print("Loaded:", model_id)
    print("pad_token_id:", tok.pad_token_id, "eos_token_id:", tok.eos_token_id)
    print("model max_position_embeddings:", getattr(model.config, "max_position_embeddings", None))
    return tok, model

llm_tokenizer, llm = load_local_llm(MODEL_ID)


Loading tokenizer: mistralai/Mistral-7B-Instruct-v0.2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.2
pad_token_id: 2 eos_token_id: 2
model max_position_embeddings: 32768


In [8]:

# ============================================================
# 7. Pairwise prompt and generation helpers
# ============================================================
SYSTEM_PROMPT_ASAP = """
You are an expert English exam rater.

Your task is to compare two student essays using the given prompt and rubric, then decide which essay would receive the higher overall rubric score.

The order of the essays is random. Do not favor Essay 1 or Essay 2 because of position. Base the decision only on the prompt, rubric, and essay quality.

Choose "tie" only when the two essays would likely receive the same final rubric score. Do not choose "tie" merely because both essays have strengths and weaknesses. If one essay is even slightly more likely to receive a higher score, choose that essay.

Return only one valid JSON object.
""".strip()


def trim_essay_for_llm(text: str, max_tokens: int) -> str:
    """Trim only the student essay. Keep rubric/source prompt intact."""
    text = "" if text is None else str(text)
    ids = llm_tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text

    head_n = int(max_tokens * 0.65)
    tail_n = max_tokens - head_n
    head = llm_tokenizer.decode(ids[:head_n], skip_special_tokens=True)
    tail = llm_tokenizer.decode(ids[-tail_n:], skip_special_tokens=True)
    return head + "\n\n[... middle of student essay omitted due to context limit ...]\n\n" + tail


def make_chat_text(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_ASAP},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(llm_tokenizer, "apply_chat_template") and llm_tokenizer.chat_template is not None:
        return llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return SYSTEM_PROMPT_ASAP + "\n\n" + user_prompt


def build_pairwise_user_prompt(essay_set: int, essay1: str, essay2: str) -> str:
    task = ASAP_TASKS[str(essay_set)]
    base_budget = ESSAY_TOKEN_BUDGET_BY_SET.get(essay_set, 700)

    candidate_budgets = [base_budget, 800, 650, 500, 350, 250, 150]
    candidate_budgets = (
        [b for b in candidate_budgets if b <= base_budget]
        + [b for b in candidate_budgets if b > base_budget]
    )

    seen = set()
    candidate_budgets = [b for b in candidate_budgets if not (b in seen or seen.add(b))]

    last_len = None

    for budget in candidate_budgets:
        e1 = trim_essay_for_llm(essay1, budget)
        e2 = trim_essay_for_llm(essay2, budget)

        user_prompt = f"""
# Prompt
{task['prompt']}

# Rubric Guidelines
{task['rubric']}

# Note
I have made an effort to remove personally identifying information from the essays using the Named Entity Recognizer (NER).
The relevant entities are identified in the text and then replaced with a string such as "PERSON", "ORGANIZATION", "LOCATION", "DATE", "TIME", "MONEY", "PERCENT", "CAPS" (any capitalized word) and "NUM" (any digits). Please do not penalize the essay because of the anonymizations.

# Essay1
{e1}

# Essay2
{e2}

Compare Essay1 and Essay2 according to the prompt and rubric.

Decision rules:
- Choose "essay1" if Essay1 would receive a higher overall rubric score.
- Choose "essay2" if Essay2 would receive a higher overall rubric score.
- Choose "tie" only if both essays would likely receive the same final rubric score.
- Do not choose "tie" just because both essays have some good and bad points.
- Do not favor the essay that appears first. The order is random.

Return exactly one valid JSON object in this format:
{{ "reasoning": "One brief sentence explaining the key rubric-based difference.", "preference": "essay1" or "essay2" or "tie" }}
""".strip()

        chat_text = make_chat_text(user_prompt)
        token_len = len(llm_tokenizer(chat_text, add_special_tokens=False)["input_ids"])
        last_len = token_len

        if token_len <= LLM_MAX_INPUT_TOKENS:
            return user_prompt

    raise ValueError(
        f"Prompt for essay_set={essay_set} is still too long after trimming essays. "
        f"Last token length={last_len}, LLM_MAX_INPUT_TOKENS={LLM_MAX_INPUT_TOKENS}. "
        "Increase LLM_MAX_INPUT_TOKENS or shorten the rubric."
    )


def extract_preference(raw_text: str) -> Dict[str, Any]:
    raw = "" if raw_text is None else str(raw_text).strip()
    lower = raw.lower()
    preference = None
    parsed_json = None

    # Try JSON object extraction first.
    json_match = re.search(r"\{.*?\}", raw, flags=re.DOTALL)
    if json_match:
        candidate = json_match.group(0)
        try:
            parsed_json = json.loads(candidate)
            pref = str(parsed_json.get("preference", "")).lower().strip()
            pref = pref.replace(" ", "")
            if pref in {"essay1", "essay2", "tie"}:
                preference = pref
        except Exception:
            parsed_json = None

    # Fallback: explicit preference/decision field in non-strict text.
    if preference is None:
        m = re.search(r"(?:preference|decision|final decision|answer)\s*[:=\-]\s*[\"']?\s*(essay\s*1|essay1|essay\s*2|essay2|tie)", lower)
        if m:
            preference = m.group(1).replace(" ", "")

    # Fallback: simple statements.
    if preference is None:
        if re.search(r"\btie\b|same score|equally good|equal quality", lower):
            preference = "tie"
        elif re.search(r"essay\s*1\s+(?:is\s+)?(?:better|higher|stronger|preferred|scores higher)", lower):
            preference = "essay1"
        elif re.search(r"essay\s*2\s+(?:is\s+)?(?:better|higher|stronger|preferred|scores higher)", lower):
            preference = "essay2"

    return {
        "preference": preference,
        "reasoning": "" if not isinstance(parsed_json, dict) else str(parsed_json.get("reasoning", "")),
        "raw_text": raw,
        "parse_failed": preference not in {"essay1", "essay2", "tie"},
    }


@torch.inference_mode()
def call_local_llm(user_prompts: List[str], max_new_tokens: int = MAX_NEW_TOKENS, temperature: float = GEN_TEMPERATURE) -> List[Dict[str, Any]]:
    input_texts = [make_chat_text(p) for p in user_prompts]
    inputs = llm_tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )
    prompt_len = inputs["input_ids"].shape[1]
    if prompt_len > LLM_MAX_INPUT_TOKENS:
        raise ValueError(f"Batch prompt length {prompt_len} exceeds LLM_MAX_INPUT_TOKENS={LLM_MAX_INPUT_TOKENS}.")

    model_device = next(llm.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    do_sample = temperature is not None and temperature > 0
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "use_cache": True,
        "pad_token_id": llm_tokenizer.pad_token_id,
        "eos_token_id": llm_tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = temperature

    output_ids = llm.generate(**inputs, **generation_kwargs)
    generated_ids = output_ids[:, prompt_len:]
    texts = llm_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    results = [extract_preference(t) for t in texts]

    del inputs, output_ids, generated_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


In [9]:

# ============================================================
# 8. Smoke test one pair before running all sets
# ============================================================
_smoke_set = RUN_ESSAY_SETS[0]
_smoke_df = prepared_sets[_smoke_set]["df"]
_smoke_prompt = build_pairwise_user_prompt(
    _smoke_set,
    _smoke_df.iloc[0]["essay"],
    _smoke_df.iloc[1]["essay"],
)
print("Smoke prompt chars:", len(_smoke_prompt))
print("Smoke prompt tokens:", len(llm_tokenizer(make_chat_text(_smoke_prompt), add_special_tokens=False)["input_ids"]))
smoke_result = call_local_llm([_smoke_prompt], max_new_tokens=MAX_NEW_TOKENS, temperature=GEN_TEMPERATURE)
smoke_result


Smoke prompt chars: 8201
Smoke prompt tokens: 2099


[{'preference': 'essay1',
  'reasoning': "Essay1 focuses on the author's admiration for his parents, while Essay2 describes the mood as determination and happiness/sadness. Essay1 provides a clearer and more accurate description of the mood created by the author as it directly relates to the author's feelings towards his parents.",
  'raw_text': '{ "reasoning": "Essay1 focuses on the author\'s admiration for his parents, while Essay2 describes the mood as determination and happiness/sadness. Essay1 provides a clearer and more accurate description of the mood created by the author as it directly relates to the author\'s feelings towards his parents.",\n"preference": "essay1" }',
  'parse_failed': False}]

In [10]:
_smoke_prompt_reverse = build_pairwise_user_prompt(
    _smoke_set,
    _smoke_df.iloc[1]["essay"],
    _smoke_df.iloc[0]["essay"],
)

smoke_reverse = call_local_llm(
    [_smoke_prompt_reverse],
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=GEN_TEMPERATURE,
)

print(smoke_reverse)

[{'preference': 'essay1', 'reasoning': 'Essay1 provides a more comprehensive description of the mood created in the memoir, including both determination and happiness/sadness, while Essay2 focuses solely on the mood of admiration.', 'raw_text': '{ "reasoning": "Essay1 provides a more comprehensive description of the mood created in the memoir, including both determination and happiness/sadness, while Essay2 focuses solely on the mood of admiration.", "preference": "essay1" }', 'parse_failed': False}]


In [11]:

# ============================================================
# 9. Generate pairwise judgments with debias and resume cache
# ============================================================
def pref_to_c(pref: Optional[str]) -> float:
    if pref == "essay1":
        return 1.0
    if pref == "essay2":
        return 0.0
    if pref == "tie":
        return 0.5
    return np.nan


def c_to_label(c: float) -> int:
    if np.isclose(c, 1.0):
        return 1
    if np.isclose(c, 0.0):
        return -1
    return 0


def debiased_label(forward_pref: Optional[str], reverse_pref: Optional[str]) -> Tuple[float, bool]:
    """Return c_ij in {1, 0.5, 0}; second value indicates consistency."""
    c_ij = pref_to_c(forward_pref)
    c_ji = pref_to_c(reverse_pref)
    if np.isfinite(c_ij) and np.isfinite(c_ji) and np.isclose(c_ij, 1.0 - c_ji):
        return float(c_ij), True
    return 0.5, False


def inspect_judgments(judgments: pd.DataFrame, essay_set: int):
    print(f"P{essay_set} judgments:", judgments.shape)
    if len(judgments) == 0:
        return
    print("label counts:")
    print(judgments["label"].value_counts(dropna=False).sort_index())
    print("label_float counts:")
    print(judgments["label_float"].value_counts(dropna=False).sort_index())
    if "consistent" in judgments.columns:
        print("consistent rate:", judgments["consistent"].mean())
    if "forward_parse_failed" in judgments.columns:
        print("forward parse fail rate:", judgments["forward_parse_failed"].mean())
        print("reverse parse fail rate:", judgments["reverse_parse_failed"].mean())


def generate_pairwise_judgments_for_set(
    essay_set: int,
    df_set: pd.DataFrame,
    pairs_df: pd.DataFrame,
    batch_size: int = PAIRWISE_BATCH_SIZE,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = GEN_TEMPERATURE,
) -> pd.DataFrame:
    cache_path = OUTPUT_DIR / "judgments" / f"p{essay_set}_{EXPERIMENT_TAG}_judgments.csv"
    essay_lookup = dict(zip(df_set["essay_id"].astype(int), df_set["essay"].astype(str)))

    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        done = set(zip(cached["essay_id_1"].astype(int), cached["essay_id_2"].astype(int)))
        print(f"P{essay_set}: loaded {len(cached)} cached judgments from {cache_path.name}")
    else:
        cached = pd.DataFrame()
        done = set()

    target_pairs = [
        (int(a), int(b))
        for a, b in pairs_df[["essay_id_1", "essay_id_2"]].itertuples(index=False, name=None)
        if (int(a), int(b)) not in done
    ]
    print(f"P{essay_set}: remaining pairs to judge = {len(target_pairs)} / {len(pairs_df)}")

    if len(target_pairs) == 0:
        inspect_judgments(cached, essay_set)
        return cached

    header_needed = not cache_path.exists()
    for start in tqdm(range(0, len(target_pairs), batch_size), desc=f"P{essay_set} LLM pairs"):
        batch_pairs = target_pairs[start:start + batch_size]
        forward_prompts = []
        reverse_prompts = []
        for eid1, eid2 in batch_pairs:
            e1 = essay_lookup[eid1]
            e2 = essay_lookup[eid2]
            forward_prompts.append(build_pairwise_user_prompt(essay_set, e1, e2))
            reverse_prompts.append(build_pairwise_user_prompt(essay_set, e2, e1))

        forward_results = call_local_llm(forward_prompts, max_new_tokens=max_new_tokens, temperature=temperature)
        reverse_results = call_local_llm(reverse_prompts, max_new_tokens=max_new_tokens, temperature=temperature)

        rows = []
        for (eid1, eid2), fr, rr in zip(batch_pairs, forward_results, reverse_results):
            lf, consistent = debiased_label(fr["preference"], rr["preference"])
            rows.append({
                "essay_set": essay_set,
                "essay_id_1": eid1,
                "essay_id_2": eid2,
                "forward_preference": fr["preference"],
                "reverse_preference": rr["preference"],
                "label_float": lf,
                "label": c_to_label(lf),
                "consistent": consistent,
                "forward_parse_failed": fr["parse_failed"],
                "reverse_parse_failed": rr["parse_failed"],
                "forward_reasoning": fr["reasoning"],
                "reverse_reasoning": rr["reasoning"],
                "forward_raw_text": fr["raw_text"],
                "reverse_raw_text": rr["raw_text"],
            })

        batch_df = pd.DataFrame(rows)
        batch_df.to_csv(cache_path, mode="a", index=False, header=header_needed)
        header_needed = False

    judgments = pd.read_csv(cache_path)
    inspect_judgments(judgments, essay_set)
    return judgments

all_judgments_by_set = {}
for essay_set in RUN_ESSAY_SETS:
    all_judgments_by_set[essay_set] = generate_pairwise_judgments_for_set(
        essay_set=essay_set,
        df_set=prepared_sets[essay_set]["df"],
        pairs_df=prepared_sets[essay_set]["pairs"],
        batch_size=PAIRWISE_BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
    )


P5: loaded 5000 cached judgments from p5_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_judgments.csv
P5: remaining pairs to judge = 0 / 5000
P5 judgments: (5000, 14)
label counts:
label
-1    1926
 0    1216
 1    1858
Name: count, dtype: int64
label_float counts:
label_float
0.0    1926
0.5    1216
1.0    1858
Name: count, dtype: int64
consistent rate: 0.7568
forward parse fail rate: 0.0002
reverse parse fail rate: 0.0002
P6: loaded 5000 cached judgments from p6_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_judgments.csv
P6: remaining pairs to judge = 0 / 5000
P6 judgments: (5000, 14)
label counts:
label
-1    1874
 0    1242
 1    1884
Name: count, dtype: int64
label_float counts:
label_float
0.0    1874
0.5    1242
1.0    1884
Name: count, dtype: int64
consistent rate: 0.7518
forward parse fail rate: 0.0002
reverse parse fail rate: 0.001
P7: loaded 1408 cached judgments from p7_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_judgments.csv
P7: remaining pairs to judge = 359

P7 LLM pairs:   0%|          | 0/38 [00:00<?, ?it/s]

P7 judgments: (5000, 14)
label counts:
label
-1    1905
 0    1166
 1    1929
Name: count, dtype: int64
label_float counts:
label_float
0.0    1905
0.5    1166
1.0    1929
Name: count, dtype: int64
consistent rate: 0.7668
forward parse fail rate: 0.0046
reverse parse fail rate: 0.0064
P8: remaining pairs to judge = 5000 / 5000


P8 LLM pairs:   0%|          | 0/53 [00:00<?, ?it/s]

P8 judgments: (5000, 14)
label counts:
label
-1     966
 0    3067
 1     967
Name: count, dtype: int64
label_float counts:
label_float
0.0     966
0.5    3067
1.0     967
Name: count, dtype: int64
consistent rate: 0.3866
forward parse fail rate: 0.0018
reverse parse fail rate: 0.0022


In [12]:

# ============================================================
# 10. Free LLM VRAM before embedding and RankNet
# ============================================================
try:
    del llm
    del llm_tokenizer
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed pairwise LLM from memory.")


Freed pairwise LLM from memory.


In [13]:

# ============================================================
# 11. Embedding helpers: OpenAI paper embedding or RoBERTa [CLS] fallback
# ============================================================
def encode_essays_openai(df: pd.DataFrame, text_col: str = "essay", batch_size: int = 64) -> np.ndarray:
    from openai import OpenAI
    client = OpenAI()
    texts = df[text_col].fillna("").astype(str).tolist()
    all_vecs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="OpenAI embeddings"):
        batch_texts = texts[start:start + batch_size]
        resp = client.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=batch_texts)
        data = sorted(resp.data, key=lambda x: x.index)
        all_vecs.extend([item.embedding for item in data])
    return np.asarray(all_vecs, dtype=np.float32)


_embed_tokenizer = None
_embed_model = None

def load_roberta_encoder():
    global _embed_tokenizer, _embed_model
    if _embed_tokenizer is None or _embed_model is None:
        print("Loading local embedding model:", LOCAL_EMBEDDING_MODEL)
        _embed_tokenizer = AutoTokenizer.from_pretrained(LOCAL_EMBEDDING_MODEL)
        _embed_model = AutoModel.from_pretrained(LOCAL_EMBEDDING_MODEL).to(DEVICE)
        _embed_model.eval()
    return _embed_tokenizer, _embed_model


@torch.inference_mode()
def encode_essays_roberta_cls(df: pd.DataFrame, text_col: str = "essay", batch_size: int = LOCAL_EMBEDDING_BATCH_SIZE) -> np.ndarray:
    tok, model = load_roberta_encoder()
    texts = df[text_col].fillna("").astype(str).tolist()
    all_vecs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="RoBERTa CLS embeddings"):
        batch_texts = texts[start:start + batch_size]
        inputs = tok(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=LOCAL_EMBEDDING_MAX_LENGTH,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        outputs = model(**inputs)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vec.detach().cpu().numpy())
        del inputs, outputs, cls_vec
    return np.vstack(all_vecs).astype(np.float32)


def get_embeddings_for_set(essay_set: int, df_set: pd.DataFrame) -> np.ndarray:
    if EMBEDDING_BACKEND == "openai":
        model_tag = OPENAI_EMBEDDING_MODEL.replace("-", "_")
    elif EMBEDDING_BACKEND == "roberta_cls":
        model_tag = LOCAL_EMBEDDING_MODEL.replace("-", "_") + "_cls"
    else:
        raise ValueError(f"Unknown EMBEDDING_BACKEND={EMBEDDING_BACKEND}")

    emb_path = OUTPUT_DIR / "embeddings" / f"p{essay_set}_{model_tag}_seed{SEED}_frac{str(SAMPLE_FRAC).replace('.', '')}.npy"
    if emb_path.exists():
        emb = np.load(emb_path)
        print(f"P{essay_set}: loaded embeddings {emb.shape} from {emb_path.name}")
        return emb

    if EMBEDDING_BACKEND == "openai":
        emb = encode_essays_openai(df_set)
    else:
        emb = encode_essays_roberta_cls(df_set)

    np.save(emb_path, emb)
    print(f"P{essay_set}: saved embeddings {emb.shape} to {emb_path.name}")
    return emb


In [14]:

# ============================================================
# 12. RankNet model and training
# ============================================================
class RankNet(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)


def train_ranknet(
    df_set: pd.DataFrame,
    embeddings: np.ndarray,
    judgments: pd.DataFrame,
    epochs: int = RANKNET_EPOCHS,
    batch_size: int = RANKNET_BATCH_SIZE,
    lr: float = RANKNET_LR,
    weight_decay: float = RANKNET_WEIGHT_DECAY,
    hidden_dim: int = RANKNET_HIDDEN_DIM,
    dropout: float = RANKNET_DROPOUT,
    device: str = DEVICE,
) -> RankNet:
    essay_ids = df_set["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].astype(int).isin(id_to_idx)
        & judgments["essay_id_2"].astype(int).isin(id_to_idx)
    ].copy()
    if len(pair_df) == 0:
        raise ValueError("No valid pairwise judgments for this set.")

    idx_i = pair_df["essay_id_1"].astype(int).map(id_to_idx).to_numpy()
    idx_j = pair_df["essay_id_2"].astype(int).map(id_to_idx).to_numpy()
    labels = pair_df["label_float"].astype(float).to_numpy(dtype=np.float32)

    x_all = torch.tensor(embeddings, dtype=torch.float32, device=device)
    idx_i_t = torch.tensor(idx_i, dtype=torch.long)
    idx_j_t = torch.tensor(idx_j, dtype=torch.long)
    labels_t = torch.tensor(labels, dtype=torch.float32)

    dataset = TensorDataset(idx_i_t, idx_j_t, labels_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    model = RankNet(input_dim=embeddings.shape[1], hidden_dim=hidden_dim, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for bi, bj, y in loader:
            bi = bi.to(device)
            bj = bj.to(device)
            y = y.to(device)

            s_i = model(x_all[bi])
            s_j = model(x_all[bj])
            logits = s_i - s_j
            loss = loss_fn(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch {epoch:03d} | loss={np.mean(losses):.4f}")

    return model


@torch.no_grad()
def predict_latent_scores(model: RankNet, embeddings: np.ndarray, device: str = DEVICE) -> np.ndarray:
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


def map_latent_to_scores(latent_scores: np.ndarray, y_min: int, y_max: int) -> np.ndarray:
    s_min = float(np.min(latent_scores))
    s_max = float(np.max(latent_scores))
    if np.isclose(s_min, s_max):
        mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2, dtype=float)
    else:
        mapped = (latent_scores - s_min) / (s_max - s_min)
        mapped = mapped * (y_max - y_min) + y_min
    mapped = np.rint(mapped)
    mapped = np.clip(mapped, y_min, y_max)
    return mapped.astype(int)


In [15]:

# ============================================================
# 13. Train/evaluate all 8 essay sets
# ============================================================
def run_ranknet_for_set(essay_set: int) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    print("=" * 80)
    print(f"P{essay_set}: RankNet training/evaluation")
    df_set = prepared_sets[essay_set]["df"].copy().reset_index(drop=True)
    y_min, y_max = ASAP_SCORE_RANGES[essay_set]

    judgment_path = OUTPUT_DIR / "judgments" / f"p{essay_set}_{EXPERIMENT_TAG}_judgments.csv"
    if not judgment_path.exists():
        raise FileNotFoundError(f"Missing judgments: {judgment_path}")
    judgments = pd.read_csv(judgment_path)

    emb = get_embeddings_for_set(essay_set, df_set)
    assert len(emb) == len(df_set), f"Embedding rows {len(emb)} != essays {len(df_set)}"

    model = train_ranknet(
        df_set=df_set,
        embeddings=emb,
        judgments=judgments,
        epochs=RANKNET_EPOCHS,
        batch_size=RANKNET_BATCH_SIZE,
        lr=RANKNET_LR,
        weight_decay=RANKNET_WEIGHT_DECAY,
        hidden_dim=RANKNET_HIDDEN_DIM,
        dropout=RANKNET_DROPOUT,
        device=DEVICE,
    )

    latent = predict_latent_scores(model, emb, device=DEVICE)
    pred = map_latent_to_scores(latent, y_min, y_max)

    scored = df_set[["essay_id", "essay_set", "split", "domain1_score", "essay"]].copy()
    scored["latent_score"] = latent
    scored["pred_score"] = pred
    scored["abs_error"] = np.abs(scored["domain1_score"] - scored["pred_score"])

    pred_path = OUTPUT_DIR / "predictions" / f"p{essay_set}_{EXPERIMENT_TAG}_{EMBEDDING_BACKEND}_predictions.csv"
    scored.to_csv(pred_path, index=False)
    print("Saved predictions:", pred_path)

    all_metrics = compute_metrics(scored["domain1_score"], scored["pred_score"])
    label_counts = judgments["label"].value_counts().to_dict()
    parse_fail_rate = float((judgments["forward_parse_failed"].mean() + judgments["reverse_parse_failed"].mean()) / 2.0)
    consistency_rate = float(judgments["consistent"].mean()) if "consistent" in judgments.columns else np.nan

    summary = {
        "essay_set": essay_set,
        "n_essays": len(scored),
        "n_pairs": len(judgments),
        "score_min": y_min,
        "score_max": y_max,
        "embedding_backend": EMBEDDING_BACKEND,
        "QWK_all": all_metrics["QWK"],
        "Spearman_all": all_metrics["Spearman"],
        "MAE_all": all_metrics["MAE"],
        "RMSE_all": all_metrics["RMSE"],
        "consistency_rate": consistency_rate,
        "parse_fail_rate": parse_fail_rate,
        "label_win_essay1": int(label_counts.get(1, 0)),
        "label_win_essay2": int(label_counts.get(-1, 0)),
        "label_tie": int(label_counts.get(0, 0)),
        "prediction_path": str(pred_path),
    }

    split_rows = []
    for split_name, g in scored.groupby("split"):
        if len(g) >= 2 and g["domain1_score"].nunique() > 1 and g["pred_score"].nunique() > 1:
            m = compute_metrics(g["domain1_score"], g["pred_score"])
        else:
            m = {"QWK": np.nan, "MAE": np.nan, "RMSE": np.nan, "Spearman": np.nan}
        split_rows.append({"essay_set": essay_set, "split": split_name, "n": len(g), **m})
    split_metrics = pd.DataFrame(split_rows)
    split_path = OUTPUT_DIR / "predictions" / f"p{essay_set}_{EXPERIMENT_TAG}_{EMBEDDING_BACKEND}_split_metrics.csv"
    split_metrics.to_csv(split_path, index=False)

    # Free per-set RankNet tensors.
    del model, emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("P{} all metrics:".format(essay_set), all_metrics)
    return scored, summary

all_summaries = []
for essay_set in RUN_ESSAY_SETS:
    _, summary = run_ranknet_for_set(essay_set)
    all_summaries.append(summary)

metrics_summary = pd.DataFrame(all_summaries).sort_values("essay_set")
metrics_summary_path = OUTPUT_DIR / f"asap8_{EXPERIMENT_TAG}_{EMBEDDING_BACKEND}_metrics_summary.csv"
metrics_summary.to_csv(metrics_summary_path, index=False)
print("Saved summary:", metrics_summary_path)
metrics_summary


P5: RankNet training/evaluation
Loading local embedding model: roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

RoBERTa CLS embeddings:   0%|          | 0/10 [00:00<?, ?it/s]

P7: saved embeddings (157, 768) to p7_roberta_base_cls_seed42_frac01.npy
Epoch 001 | loss=0.6924
Epoch 010 | loss=0.6323
Epoch 020 | loss=0.5570
Epoch 030 | loss=0.5027
Epoch 040 | loss=0.4787
Epoch 050 | loss=0.4547
Epoch 060 | loss=0.4437
Epoch 070 | loss=0.4398
Epoch 080 | loss=0.4284
Epoch 090 | loss=0.4250
Epoch 100 | loss=0.4195
Saved predictions: /content/lces_asap8_outputs/predictions/p7_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_roberta_cls_predictions.csv
P7 all metrics: {'QWK': 0.674638321544382, 'MAE': 3.4585987261146496, 'RMSE': 4.370573283680927, 'Spearman': 0.7408492765462911}
P8: RankNet training/evaluation


RoBERTa CLS embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

P8: saved embeddings (72, 768) to p8_roberta_base_cls_seed42_frac01.npy
Epoch 001 | loss=0.6932
Epoch 010 | loss=0.6792
Epoch 020 | loss=0.6579
Epoch 030 | loss=0.6407
Epoch 040 | loss=0.6299
Epoch 050 | loss=0.6227
Epoch 060 | loss=0.6157
Epoch 070 | loss=0.6108
Epoch 080 | loss=0.6056
Epoch 090 | loss=0.6022
Epoch 100 | loss=0.6071
Saved predictions: /content/lces_asap8_outputs/predictions/p8_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_roberta_cls_predictions.csv
P8 all metrics: {'QWK': 0.4358477814457643, 'MAE': 8.277777777777779, 'RMSE': 10.357498625526231, 'Spearman': 0.4968950037338272}
Saved summary: /content/lces_asap8_outputs/asap8_Mistral_7B_Instruct_v02_temp0_m5000_seed42_frac01_roberta_cls_metrics_summary.csv


,essay_set,n_essays,n_pairs,score_min,score_max,embedding_backend,QWK_all,Spearman_all,MAE_all,RMSE_all,consistency_rate,parse_fail_rate,label_win_essay1,label_win_essay2,label_tie,prediction_path
0,5,180,5000,0,4,roberta_cls,0.770984,0.750908,0.400000,0.632456,0.7568,0.0002,1858,1926,1216,/content/lces_asap8_outputs/predictions/p5_Mis...
1,6,180,5000,0,4,roberta_cls,0.692158,0.733982,0.494444,0.726483,0.7518,0.0006,1884,1874,1242,/content/lces_asap8_outputs/predictions/p6_Mis...
2,7,157,5000,0,30,roberta_cls,0.674638,0.740849,3.458599,4.370573,0.7668,0.0055,1929,1905,1166,/content/lces_asap8_outputs/predictions/p7_Mis...
3,8,72,5000,0,60,roberta_cls,0.435848,0.496895,8.277778,10.357499,0.3866,0.0020,967,966,3067,/content/lces_asap8_outputs/predictions/p8_Mis...


In [16]:

# ============================================================
# 14. Show compact summary and package outputs
# ============================================================
compact_cols = [
    "essay_set", "n_essays", "n_pairs", "score_min", "score_max",
    "QWK_all", "Spearman_all", "MAE_all", "RMSE_all",
    "consistency_rate", "parse_fail_rate", "label_tie",
]
compact_summary = metrics_summary[compact_cols].copy()
compact_summary.loc["AVG"] = {
    "essay_set": "AVG",
    "n_essays": compact_summary["n_essays"].sum(),
    "n_pairs": compact_summary["n_pairs"].sum(),
    "score_min": np.nan,
    "score_max": np.nan,
    "QWK_all": compact_summary["QWK_all"].mean(),
    "Spearman_all": compact_summary["Spearman_all"].mean(),
    "MAE_all": compact_summary["MAE_all"].mean(),
    "RMSE_all": compact_summary["RMSE_all"].mean(),
    "consistency_rate": compact_summary["consistency_rate"].mean(),
    "parse_fail_rate": compact_summary["parse_fail_rate"].mean(),
    "label_tie": compact_summary["label_tie"].sum(),
}
compact_summary


,essay_set,n_essays,n_pairs,score_min,score_max,QWK_all,Spearman_all,MAE_all,RMSE_all,consistency_rate,parse_fail_rate,label_tie
0,5,180,5000,0.0,4.0,0.770984,0.750908,0.400000,0.632456,0.7568,0.000200,1216
1,6,180,5000,0.0,4.0,0.692158,0.733982,0.494444,0.726483,0.7518,0.000600,1242
2,7,157,5000,0.0,30.0,0.674638,0.740849,3.458599,4.370573,0.7668,0.005500,1166
3,8,72,5000,0.0,60.0,0.435848,0.496895,8.277778,10.357499,0.3866,0.002000,3067
AVG,AVG,589,20000,NaN,NaN,0.643407,0.680659,3.157705,4.021753,0.6655,0.002075,6691


In [17]:

# ============================================================
# 15. Zip outputs for download
# ============================================================
zip_base = "/content/lces_asap8_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Not running in Colab or download failed. You can find the zip at:", zip_path)
    print("Error:", e)


Created: /content/lces_asap8_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# ============================================================
# Auto disconnect runtime after 300 seconds
# ============================================================

import time
import os

WAIT_SECONDS = 300

print(f"Runtime will disconnect in {WAIT_SECONDS} seconds...")
time.sleep(WAIT_SECONDS)

print("Disconnecting runtime now...")

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("runtime.unassign() failed, force killing process:", repr(e))
    os.kill(os.getpid(), 9)

Runtime will disconnect in 300 seconds...


KeyboardInterrupt: 